# Chapter 9 — Public Python Interfaces, Omega CLI, and Local Run Evidence

The final increment turns the cumulative harness into one reusable, observable product without moving terminal behavior into Runtime.

## Goal and Previous Limitation

Chapter 8 has a complete Coding Agent seam, but callers still assemble infrastructure themselves and there is no stable RunResult, machine event envelope, local Run catalog, or installed Omega interface.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
CHAPTER_8 = ROOT / 'course' / 'checkpoints' / 'ch08'
if not (CHAPTER_8 / 'checkpoint.json').is_file():
    raise RuntimeError('run this Chapter Notebook from the repository root')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

## Conceptual Model

Factories assemble explicit adapters and policies; AgentSession remains the control interface; RunObserver translates passive Events into versioned local evidence; terminal and non-terminal adapters render the same snapshots, queues, cancellation, commands, and results.

## Minimal Execution

The Export Cells below replace complete modules. A temporary import then proves the high-level no-save path without credentials, disk evidence, or network access.

In [ ]:
RUNTIME_SOURCE = '"""Async Agent Runtime with structured Tool batches."""\n\nfrom __future__ import annotations\n\nimport asyncio\nfrom collections.abc import AsyncIterator, Awaitable, Callable, Mapping, Sequence\nfrom dataclasses import dataclass, replace\nfrom enum import Enum\nimport hashlib\nimport json\nfrom types import MappingProxyType\nfrom typing import Protocol, TypeAlias, cast\nfrom uuid import uuid4\n\nfrom jsonschema import (  # type: ignore[import-untyped]\n    Draft202012Validator,\n    ValidationError,\n)\n\nfrom .model import (\n    AgentMessage,\n    ContentBlock,\n    ModelAdapter,\n    ModelAdapterError,\n    ModelEnd,\n    ModelError,\n    ModelErrorCode,\n    ModelEvent,\n    ModelRequest,\n    ModelOperation,\n    ModelSpec,\n    Role,\n    StopReason,\n    TextContent,\n    TextDelta,\n    ToolCallContent,\n    ToolCallDelta,\n    Usage,\n    UsageUpdate,\n    to_model_messages,\n)\nfrom .compaction import CompactionCheckpoint\nfrom .extensions import (\n    HookContext,\n    HookExecutionError,\n    HookPoint,\n    HookRegistration,\n    SubscriberRegistration,\n)\nfrom .tools import (\n    LocalToolExecutor,\n    PreparedToolCall,\n    Tool,\n    ToolErrorCode,\n    ToolExecutor,\n    ToolOutputBudget,\n    ToolResult,\n    ToolResultMessage,\n    bound_tool_result,\n)\n\n\nclass EventType(str, Enum):\n    AGENT_START = "agent_start"\n    MODEL_ATTEMPT_START = "model_attempt_start"\n    MODEL_EVENT = "model_event"\n    MODEL_ATTEMPT_FAILED = "model_attempt_failed"\n    RETRY_SCHEDULED = "retry_scheduled"\n    COMPACTION_START = "compaction_start"\n    COMPACTION_END = "compaction_end"\n    COMPACTION_FAILED = "compaction_failed"\n    TOOL_BATCH_START = "tool_batch_start"\n    TOOL_CALL_START = "tool_call_start"\n    TOOL_CALL_END = "tool_call_end"\n    TOOL_BATCH_END = "tool_batch_end"\n    RUN_CANCELLED = "run_cancelled"\n    MESSAGE_END = "message_end"\n    AGENT_END = "agent_end"\n    SUBSCRIBER_FAILED = "subscriber_failed"\n\n\nclass TerminalStatus(str, Enum):\n    COMPLETED = "completed"\n    MODEL_ERROR = "model_error"\n    CANCELLED = "cancelled"\n    MAX_TURNS = "max_turns"\n    MAX_TOOL_CALLS = "max_tool_calls"\n    TIMEOUT = "timeout"\n    MAX_TOTAL_TOKENS = "max_total_tokens"\n\n\n@dataclass(frozen=True, slots=True)\nclass RunGuard:\n    max_turns: int | None = None\n    max_tool_calls: int | None = None\n    timeout_seconds: float | None = None\n    max_total_tokens: int | None = None\n\n    def __post_init__(self) -> None:\n        integer_limits = {\n            "max_turns": self.max_turns,\n            "max_tool_calls": self.max_tool_calls,\n            "max_total_tokens": self.max_total_tokens,\n        }\n        for name, value in integer_limits.items():\n            if value is not None and (\n                isinstance(value, bool) or not isinstance(value, int) or value <= 0\n            ):\n                raise ValueError(f"{name} must be a positive integer when supplied")\n        if self.timeout_seconds is not None and self.timeout_seconds <= 0:\n            raise ValueError("timeout_seconds must be positive when supplied")\n\n    def reached(\n        self,\n        *,\n        turns: int,\n        tool_calls: int,\n        usage: Usage | None,\n        elapsed_seconds: float,\n    ) -> TerminalStatus | None:\n        if (\n            self.timeout_seconds is not None\n            and elapsed_seconds >= self.timeout_seconds\n        ):\n            return TerminalStatus.TIMEOUT\n        if self.max_turns is not None and turns >= self.max_turns:\n            return TerminalStatus.MAX_TURNS\n        if (\n            self.max_tool_calls is not None\n            and tool_calls >= self.max_tool_calls\n        ):\n            return TerminalStatus.MAX_TOOL_CALLS\n        if (\n            self.max_total_tokens is not None\n            and usage is not None\n            and usage.total_tokens >= self.max_total_tokens\n        ):\n            return TerminalStatus.MAX_TOTAL_TOKENS\n        return None\n\n\n@dataclass(frozen=True, slots=True)\nclass RuntimeEvent:\n    sequence: int\n    type: EventType\n    attempt: int | None = None\n    model_event: ModelEvent | None = None\n    error: ModelError | None = None\n    retry_delay_seconds: float | None = None\n    partial_text: str = ""\n    partial_usage: Usage | None = None\n    tool_call_id: str | None = None\n    tool_name: str | None = None\n    tool_arguments: Mapping[str, object] | None = None\n    tool_result: ToolResult | None = None\n    operation: ModelOperation = ModelOperation.RUN\n    extension_name: str | None = None\n    diagnostic: str | None = None\n    run_id: str | None = None\n    snapshot_fingerprint: str | None = None\n\n\n@dataclass(frozen=True, slots=True)\nclass AssistantOutcome:\n    message: AgentMessage\n    stop_reason: StopReason\n    usage: Usage | None = None\n    error: ModelError | None = None\n    attempts: int = 1\n    tool_results: tuple[ToolResult, ...] = ()\n    status: TerminalStatus = TerminalStatus.COMPLETED\n    snapshot_fingerprint: str | None = None\n\n\n@dataclass(frozen=True, slots=True)\nclass SummaryGeneration:\n    text: str\n    usage: Usage | None\n    attempts: int\n\n\n@dataclass(frozen=True, slots=True)\nclass RetryPolicy:\n    delays: tuple[float, ...] = (2.0, 4.0, 8.0)\n    max_retry_after_seconds: float = 60.0\n\n    def __post_init__(self) -> None:\n        if any(delay < 0 for delay in self.delays):\n            raise ValueError("retry delays cannot be negative")\n        if self.max_retry_after_seconds < 0:\n            raise ValueError("max_retry_after_seconds cannot be negative")\n\n    def delay_for(\n        self,\n        error: ModelError,\n        failed_attempt: int,\n        *,\n        retry_after_seconds: float | None = None,\n    ) -> float | None:\n        retryable_codes = {\n            ModelErrorCode.RATE_LIMIT,\n            ModelErrorCode.TIMEOUT,\n            ModelErrorCode.CONNECTION,\n            ModelErrorCode.SERVER,\n        }\n        retryable_status = error.status_code in {408, 429} or (\n            error.status_code is not None and error.status_code >= 500\n        )\n        if (\n            error.code not in retryable_codes and not retryable_status\n        ) or failed_attempt > len(self.delays):\n            return None\n        if retry_after_seconds is not None:\n            if not 0 <= retry_after_seconds <= self.max_retry_after_seconds:\n                return None\n            return retry_after_seconds\n        return self.delays[failed_attempt - 1]\n\n\n@dataclass(frozen=True, slots=True)\nclass RunSnapshot:\n    """Immutable effective behavior captured when a Run is accepted."""\n\n    run_id: str\n    adapter: ModelAdapter\n    model: ModelSpec\n    tools: tuple[Tool, ...]\n    tool_executor: ToolExecutor\n    tool_output_budget: ToolOutputBudget\n    retry_policy: RetryPolicy\n    sleeper: Sleeper\n    run_guard: object | None\n    extension_identities: tuple[str, ...]\n    subscribers: tuple[SubscriberRegistration, ...]\n    hooks: tuple[HookRegistration, ...]\n    generation_settings: Mapping[str, object]\n    compaction_policy: object | None\n    prompt_hashes: tuple[tuple[str, str], ...]\n    resource_hashes: tuple[tuple[str, str], ...]\n    fingerprint: str\n\n    @classmethod\n    def capture(\n        cls,\n        *,\n        adapter: ModelAdapter,\n        model: ModelSpec,\n        tools: Sequence[Tool],\n        tool_executor: ToolExecutor,\n        tool_output_budget: ToolOutputBudget,\n        retry_policy: RetryPolicy,\n        sleeper: Sleeper,\n        run_guard: object | None,\n        extension_identities: Sequence[str] = (),\n        subscribers: Sequence[SubscriberRegistration] = (),\n        hooks: Sequence[HookRegistration] = (),\n        generation_settings: Mapping[str, object] | None = None,\n        compaction_policy: object | None = None,\n        prompt_hashes: Mapping[str, str] | None = None,\n        resource_hashes: Mapping[str, str] | None = None,\n    ) -> "RunSnapshot":\n        accepted_tools = tuple(tools)\n        accepted_extensions = tuple(extension_identities)\n        accepted_subscribers = tuple(subscribers)\n        accepted_hooks = tuple(hooks)\n        accepted_generation = dict(generation_settings or {})\n        accepted_prompt_hashes = tuple(sorted((prompt_hashes or {}).items()))\n        accepted_resource_hashes = tuple(sorted((resource_hashes or {}).items()))\n        payload = {\n            "model": {\n                "id": model.model_id,\n                "context_window": model.context_window,\n                "max_output_tokens": model.max_output_tokens,\n                "supports_tools": model.supports_tools,\n            },\n            "tools": [\n                {\n                    "name": tool.name,\n                    "description": tool.description,\n                    "schema": dict(tool.input_schema),\n                    "sequential": tool.sequential,\n                    "output_direction": tool.output_direction.value,\n                }\n                for tool in accepted_tools\n            ],\n            "extensions": list(accepted_extensions),\n            "hooks": [\n                [registration.extension_name, registration.point.value]\n                for registration in accepted_hooks\n            ],\n            "generation": accepted_generation,\n            "retry": {\n                "delays": retry_policy.delays,\n                "max_retry_after_seconds": retry_policy.max_retry_after_seconds,\n            },\n            "compaction": repr(compaction_policy),\n            "guard": repr(run_guard),\n            "prompt_hashes": accepted_prompt_hashes,\n            "resource_hashes": accepted_resource_hashes,\n        }\n        canonical = json.dumps(\n            payload,\n            ensure_ascii=False,\n            sort_keys=True,\n            separators=(",", ":"),\n        ).encode("utf-8")\n        return cls(\n            uuid4().hex,\n            adapter,\n            model,\n            accepted_tools,\n            tool_executor,\n            tool_output_budget,\n            retry_policy,\n            sleeper,\n            run_guard,\n            accepted_extensions,\n            accepted_subscribers,\n            accepted_hooks,\n            MappingProxyType(accepted_generation),\n            compaction_policy,\n            accepted_prompt_hashes,\n            accepted_resource_hashes,\n            hashlib.sha256(canonical).hexdigest(),\n        )\n\n\nSleeper: TypeAlias = Callable[[float], Awaitable[None]]\nConversationMessage: TypeAlias = AgentMessage | ToolResultMessage\nSettlementSink: TypeAlias = Callable[[Sequence[ConversationMessage]], None]\nContextOverflowRecovery: TypeAlias = Callable[[], Awaitable[bool]]\nSettledTurnHandler: TypeAlias = Callable[[], Awaitable[object]]\n_EVENTS_DONE = object()\n\n\nclass TurnInput(Protocol):\n    """Runtime-facing view of Steering Messages waiting at a turn boundary."""\n\n    def pending(self) -> bool: ...\n\n    def take(self) -> Sequence[AgentMessage]: ...\n\n\n@dataclass(slots=True)\nclass _CancellationState:\n    status: TerminalStatus = TerminalStatus.CANCELLED\n    requested: bool = False\n    started: bool = False\n\n    def request(self, status: TerminalStatus) -> bool:\n        if self.requested:\n            return False\n        self.status = status\n        self.requested = True\n        return True\n\n\ndef _add_usage(left: Usage | None, right: Usage | None) -> Usage | None:\n    if left is None:\n        return right\n    if right is None:\n        return left\n    return Usage(\n        left.input_tokens + right.input_tokens,\n        left.output_tokens + right.output_tokens,\n        left.total_tokens + right.total_tokens,\n        left.estimated or right.estimated,\n    )\n\n\nclass AgentRunHandle:\n    """One accepted run\'s observations, cancellation, and eventual outcome."""\n\n    def __init__(\n        self,\n        task: asyncio.Task[AssistantOutcome],\n        events: asyncio.Queue[RuntimeEvent | object],\n        cancellation: _CancellationState,\n        snapshot: RunSnapshot,\n    ) -> None:\n        self._task = task\n        self._events = events\n        self._cancellation = cancellation\n        self._snapshot = snapshot\n\n    @property\n    def run_id(self) -> str:\n        return self._snapshot.run_id\n\n    @property\n    def snapshot(self) -> RunSnapshot:\n        return self._snapshot\n\n    async def events(self) -> AsyncIterator[RuntimeEvent]:\n        while True:\n            event = await self._events.get()\n            if event is _EVENTS_DONE:\n                break\n            yield cast(RuntimeEvent, event)\n\n    async def result(self) -> AssistantOutcome:\n        outcome = await self._task\n        if outcome.snapshot_fingerprint == self._snapshot.fingerprint:\n            return outcome\n        return replace(\n            outcome,\n            snapshot_fingerprint=self._snapshot.fingerprint,\n        )\n\n    def cancel(self) -> None:\n        self._cancel_with(TerminalStatus.CANCELLED)\n\n    def _cancel_with(self, status: TerminalStatus) -> None:\n        if not self._task.done() and self._cancellation.request(status):\n            if self._cancellation.started:\n                self._task.cancel()\n\n\nclass AgentRuntime:\n    """Advance typed conversation state through model and Tool turns."""\n\n    def __init__(\n        self,\n        adapter: ModelAdapter,\n        model: ModelSpec,\n        *,\n        tools: Sequence[Tool] = (),\n        tool_executor: ToolExecutor | None = None,\n        tool_output_budget: ToolOutputBudget | None = None,\n        retry_policy: RetryPolicy | None = None,\n        sleeper: Sleeper = asyncio.sleep,\n        run_guard: object | None = None,\n        generation_settings: Mapping[str, object] | None = None,\n        history: Sequence[ConversationMessage] = (),\n    ) -> None:\n        if not isinstance(model, ModelSpec):\n            raise TypeError("model must be a ModelSpec")\n        if not callable(getattr(adapter, "stream", None)):\n            raise TypeError("adapter must implement ModelAdapter.stream")\n        if retry_policy is not None and not isinstance(retry_policy, RetryPolicy):\n            raise TypeError("retry_policy must be a RetryPolicy")\n        if not callable(sleeper):\n            raise TypeError("sleeper must be an async callable")\n        if tool_executor is not None and not callable(\n            getattr(tool_executor, "execute", None)\n        ):\n            raise TypeError("tool_executor must implement ToolExecutor.execute")\n        if tool_output_budget is not None and not isinstance(\n            tool_output_budget, ToolOutputBudget\n        ):\n            raise TypeError("tool_output_budget must be a ToolOutputBudget")\n        registered: dict[str, Tool] = {}\n        for tool in tools:\n            if not isinstance(tool, Tool):\n                raise TypeError("tools must contain Tool values")\n            if tool.name in registered:\n                raise ValueError(f"duplicate Tool name: {tool.name!r}")\n            registered[tool.name] = tool\n        if registered and not model.supports_tools:\n            raise ValueError("configured ModelSpec does not support Tools")\n        self._adapter = adapter\n        self._model = model\n        self._tools = registered\n        self._tool_executor = tool_executor or LocalToolExecutor()\n        self._tool_output_budget = tool_output_budget or ToolOutputBudget()\n        self._retry_policy = retry_policy or RetryPolicy()\n        self._sleeper = sleeper\n        self._run_guard = run_guard\n        self._generation_settings = MappingProxyType(dict(generation_settings or {}))\n        accepted_history = tuple(history)\n        to_model_messages(accepted_history)\n        self._history: list[ConversationMessage] = list(accepted_history)\n        self._effective_history: list[ConversationMessage] | None = None\n        self._settlement_sink: SettlementSink | None = None\n        self._context_overflow_recovery: ContextOverflowRecovery | None = None\n        self._settled_turn_handler: SettledTurnHandler | None = None\n        self._subscribers: tuple[SubscriberRegistration, ...] = ()\n        self._hooks: tuple[HookRegistration, ...] = ()\n\n    @property\n    def history(self) -> tuple[ConversationMessage, ...]:\n        return tuple(self._history)\n\n    @property\n    def effective_history(self) -> tuple[ConversationMessage, ...]:\n        return tuple(\n            self._history\n            if self._effective_history is None\n            else self._effective_history\n        )\n\n    @property\n    def model(self) -> ModelSpec:\n        return self._model\n\n    @property\n    def tools(self) -> tuple[Tool, ...]:\n        return tuple(self._tools.values())\n\n    def configure_tools(self, tools: Sequence[Tool]) -> None:\n        """Replace the effective Tool set before a Session accepts work."""\n\n        registered: dict[str, Tool] = {}\n        for tool in tools:\n            if not isinstance(tool, Tool):\n                raise TypeError("tools must contain Tool values")\n            if tool.name in registered:\n                raise ValueError(f"duplicate Tool name: {tool.name!r}")\n            registered[tool.name] = tool\n        if registered and not self._model.supports_tools:\n            raise ValueError("configured ModelSpec does not support Tools")\n        self._tools = registered\n\n    def configure_subscribers(\n        self,\n        subscribers: Sequence[SubscriberRegistration],\n    ) -> None:\n        """Install high-level passive observers before accepting a Run."""\n\n        accepted = tuple(subscribers)\n        if any(not isinstance(item, SubscriberRegistration) for item in accepted):\n            raise TypeError("subscribers must contain SubscriberRegistration values")\n        self._subscribers = accepted\n\n    def configure_hooks(self, hooks: Sequence[HookRegistration]) -> None:\n        """Install the bounded ordered Hook set before accepting a Run."""\n\n        accepted = tuple(hooks)\n        if any(not isinstance(item, HookRegistration) for item in accepted):\n            raise TypeError("hooks must contain HookRegistration values")\n        self._hooks = accepted\n\n    async def _apply_hooks(\n        self,\n        point: HookPoint,\n        value: object,\n        snapshot: RunSnapshot | None = None,\n    ) -> object:\n        current = value\n        registrations = self._hooks if snapshot is None else snapshot.hooks\n        for registration in registrations:\n            if registration.point is not point:\n                continue\n            try:\n                replacement = await registration.callback(\n                    HookContext(point, current, snapshot)\n                )\n            except Exception as error:\n                raise HookExecutionError(\n                    registration.extension_name,\n                    point,\n                    type(error).__name__,\n                ) from error\n            if replacement is not None:\n                current = replacement\n        return current\n\n    async def apply_hooks(\n        self,\n        point: HookPoint,\n        value: object,\n        *,\n        snapshot: RunSnapshot | None = None,\n    ) -> object:\n        """Invoke one frozen-order Hook point for an AgentSession operation."""\n\n        return await self._apply_hooks(point, value, snapshot)\n\n    def capture_snapshot(\n        self,\n        *,\n        extension_identities: Sequence[str] = (),\n        compaction_policy: object | None = None,\n        prompt_hashes: Mapping[str, str] | None = None,\n        resource_hashes: Mapping[str, str] | None = None,\n    ) -> RunSnapshot:\n        return RunSnapshot.capture(\n            adapter=self._adapter,\n            model=self._model,\n            tools=self.tools,\n            tool_executor=self._tool_executor,\n            tool_output_budget=self._tool_output_budget,\n            retry_policy=self._retry_policy,\n            sleeper=self._sleeper,\n            run_guard=self._run_guard,\n            extension_identities=extension_identities,\n            subscribers=self._subscribers,\n            hooks=self._hooks,\n            generation_settings=self._generation_settings,\n            compaction_policy=compaction_policy,\n            prompt_hashes=prompt_hashes,\n            resource_hashes=resource_hashes,\n        )\n\n    @property\n    def run_guard(self) -> object | None:\n        return self._run_guard\n\n    def restore_history(\n        self,\n        history: Sequence[ConversationMessage],\n        *,\n        effective_history: Sequence[ConversationMessage] | None = None,\n    ) -> None:\n        """Seed a newly constructed Runtime from one settled Session branch."""\n\n        if self._history:\n            raise RuntimeError("Runtime history must be empty before restoration")\n        accepted = tuple(history)\n        to_model_messages(accepted)\n        self._history.extend(accepted)\n        if effective_history is not None:\n            effective = tuple(effective_history)\n            to_model_messages(effective)\n            self._effective_history = list(effective)\n\n    def install_compaction(self, checkpoint: CompactionCheckpoint) -> None:\n        """Replace only the model-facing prefix at a Settled Boundary."""\n\n        self._effective_history = [\n            checkpoint.summary.as_message(),\n            *checkpoint.retained_tail,\n        ]\n\n    def set_settlement_sink(self, sink: SettlementSink | None) -> None:\n        """Install the AgentSession-owned persistence barrier for the next Run."""\n\n        if sink is not None and not callable(sink):\n            raise TypeError("settlement sink must be callable")\n        self._settlement_sink = sink\n\n    def set_context_overflow_recovery(\n        self,\n        recovery: ContextOverflowRecovery | None,\n    ) -> None:\n        if recovery is not None and not callable(recovery):\n            raise TypeError("context overflow recovery must be callable")\n        self._context_overflow_recovery = recovery\n\n    def set_settled_turn_handler(\n        self,\n        handler: SettledTurnHandler | None,\n    ) -> None:\n        if handler is not None and not callable(handler):\n            raise TypeError("settled turn handler must be callable")\n        self._settled_turn_handler = handler\n\n    def _append_settled(self, messages: Sequence[ConversationMessage]) -> None:\n        accepted = tuple(messages)\n        self._history.extend(accepted)\n        if self._effective_history is not None:\n            self._effective_history.extend(accepted)\n        if accepted and self._settlement_sink is not None:\n            self._settlement_sink(accepted)\n\n    def _append_unsettled(self, message: ConversationMessage) -> None:\n        self._history.append(message)\n        if self._effective_history is not None:\n            self._effective_history.append(message)\n\n    def _mark_settled(self, messages: Sequence[ConversationMessage]) -> None:\n        accepted = tuple(messages)\n        if accepted and self._settlement_sink is not None:\n            self._settlement_sink(accepted)\n\n    def start(\n        self,\n        messages: Sequence[AgentMessage],\n        *,\n        turn_input: TurnInput | None = None,\n        snapshot: RunSnapshot | None = None,\n    ) -> AgentRunHandle:\n        accepted = tuple(messages)\n        to_model_messages((*self.effective_history, *accepted))\n        self._append_settled(accepted)\n        events: asyncio.Queue[RuntimeEvent | object] = asyncio.Queue()\n        cancellation = _CancellationState()\n        accepted_snapshot = snapshot or self.capture_snapshot()\n        if not isinstance(accepted_snapshot, RunSnapshot):\n            raise TypeError("snapshot must be a RunSnapshot")\n        task = asyncio.get_running_loop().create_task(\n            self._execute(events, cancellation, turn_input, accepted_snapshot)\n        )\n        handle = AgentRunHandle(task, events, cancellation, accepted_snapshot)\n        if (\n            isinstance(accepted_snapshot.run_guard, RunGuard)\n            and accepted_snapshot.run_guard.timeout_seconds is not None\n        ):\n            timer = asyncio.get_running_loop().call_later(\n                accepted_snapshot.run_guard.timeout_seconds,\n                handle._cancel_with,\n                TerminalStatus.TIMEOUT,\n            )\n            task.add_done_callback(lambda completed: timer.cancel())\n        return handle\n\n    async def run(self, messages: Sequence[AgentMessage]) -> AssistantOutcome:\n        return await self.start(messages).result()\n\n    async def generate_summary(\n        self,\n        messages: Sequence[ConversationMessage],\n        *,\n        focus: str | None = None,\n        snapshot: RunSnapshot | None = None,\n    ) -> SummaryGeneration:\n        """Run a retryable model operation without mutating conversation history."""\n\n        accepted = tuple(messages)\n        if not accepted:\n            raise ValueError("summary generation requires source history")\n        effective = snapshot or self.capture_snapshot()\n        instruction = (\n            "Summarize the following settled conversation as a durable context "\n            "checkpoint. Preserve decisions, constraints, unresolved work, and "\n            "facts needed to continue. Return summary text only."\n        )\n        if focus is not None:\n            instruction += f"\\nFocus requested by the caller: {focus}"\n        summary_prompt = AgentMessage.text(Role.SYSTEM, instruction)\n        attempt = 0\n        while True:\n            attempt += 1\n            request = ModelRequest(\n                messages=to_model_messages((summary_prompt, *accepted)),\n                model=effective.model,\n                operation=ModelOperation.COMPACTION,\n            )\n            text_parts: list[str] = []\n            usage: Usage | None = None\n            end: ModelEnd | None = None\n            schema_error: ModelError | None = None\n            try:\n                async for event in effective.adapter.stream(request):\n                    if end is not None:\n                        schema_error = ModelError(\n                            ModelErrorCode.SCHEMA,\n                            "Compaction stream emitted data after ModelEnd",\n                            False,\n                        )\n                        break\n                    if isinstance(event, TextDelta):\n                        text_parts.append(event.text)\n                    elif isinstance(event, UsageUpdate):\n                        usage = event.usage\n                    elif isinstance(event, ModelEnd):\n                        end = event\n                    else:\n                        schema_error = ModelError(\n                            ModelErrorCode.SCHEMA,\n                            "Compaction summary must contain text only",\n                            False,\n                        )\n                        break\n            except ModelAdapterError as failure:\n                delay = effective.retry_policy.delay_for(\n                    failure.error,\n                    attempt,\n                    retry_after_seconds=failure.error.retry_after_seconds,\n                )\n                if delay is None:\n                    raise\n                await effective.sleeper(delay)\n                continue\n            if schema_error is None and end is None:\n                schema_error = ModelError(\n                    ModelErrorCode.SCHEMA,\n                    "Compaction stream did not end with ModelEnd",\n                    False,\n                )\n            if (\n                schema_error is None\n                and end is not None\n                and end.stop_reason is not StopReason.COMPLETE\n            ):\n                schema_error = ModelError(\n                    ModelErrorCode.SCHEMA,\n                    "Compaction summary did not complete successfully",\n                    False,\n                )\n            text = "".join(text_parts)\n            if schema_error is None and not text.strip():\n                schema_error = ModelError(\n                    ModelErrorCode.SCHEMA,\n                    "Compaction summary cannot be empty",\n                    False,\n                )\n            if schema_error is not None:\n                raise ModelAdapterError(schema_error)\n            return SummaryGeneration(text, usage, attempt)\n\n    async def _execute(\n        self,\n        events: asyncio.Queue[RuntimeEvent | object],\n        cancellation: _CancellationState,\n        turn_input: TurnInput | None = None,\n        snapshot: RunSnapshot | None = None,\n    ) -> AssistantOutcome:\n        cancellation.started = True\n        if snapshot is None:\n            snapshot = self.capture_snapshot()\n        tools = {tool.name: tool for tool in snapshot.tools}\n        request_history: list[ConversationMessage] = list(self.effective_history)\n        sequence = 0\n        total_attempts = 0\n        text_parts: list[str] = []\n        current_usage: Usage | None = None\n        run_usage: Usage | None = None\n        run_tool_results: list[ToolResult] = []\n        turns = 0\n        tool_calls = 0\n        started_at = asyncio.get_running_loop().time()\n        overflow_recovery_attempted = False\n\n        def append_settled(messages: Sequence[ConversationMessage]) -> None:\n            accepted = tuple(messages)\n            self._append_settled(accepted)\n            request_history.extend(accepted)\n\n        def append_unsettled(message: ConversationMessage) -> None:\n            self._append_unsettled(message)\n            request_history.append(message)\n\n        def guard_status() -> TerminalStatus | None:\n            if not isinstance(snapshot.run_guard, RunGuard):\n                return None\n            return snapshot.run_guard.reached(\n                turns=turns,\n                tool_calls=tool_calls,\n                usage=run_usage,\n                elapsed_seconds=asyncio.get_running_loop().time() - started_at,\n            )\n\n        async def emit(\n            type_: EventType,\n            *,\n            attempt: int | None = None,\n            model_event: ModelEvent | None = None,\n            error: ModelError | None = None,\n            retry_delay_seconds: float | None = None,\n            partial_text: str = "",\n            partial_usage: Usage | None = None,\n            tool_call_id: str | None = None,\n            tool_name: str | None = None,\n            tool_arguments: Mapping[str, object] | None = None,\n            tool_result: ToolResult | None = None,\n            operation: ModelOperation = ModelOperation.RUN,\n        ) -> None:\n            nonlocal sequence\n            sequence += 1\n            event = RuntimeEvent(\n                sequence=sequence,\n                type=type_,\n                attempt=attempt,\n                model_event=model_event,\n                error=error,\n                retry_delay_seconds=retry_delay_seconds,\n                partial_text=partial_text,\n                partial_usage=partial_usage,\n                tool_call_id=tool_call_id,\n                tool_name=tool_name,\n                tool_arguments=tool_arguments,\n                tool_result=tool_result,\n                operation=operation,\n                run_id=snapshot.run_id,\n                snapshot_fingerprint=snapshot.fingerprint,\n            )\n            await events.put(event)\n            for registration in snapshot.subscribers:\n                try:\n                    await registration.callback(event)\n                except Exception as subscriber_error:\n                    sequence += 1\n                    await events.put(\n                        RuntimeEvent(\n                            sequence=sequence,\n                            type=EventType.SUBSCRIBER_FAILED,\n                            attempt=attempt,\n                            operation=operation,\n                            extension_name=registration.extension_name,\n                            diagnostic=type(subscriber_error).__name__,\n                            run_id=snapshot.run_id,\n                            snapshot_fingerprint=snapshot.fingerprint,\n                        )\n                    )\n\n        async def finish(outcome: AssistantOutcome) -> AssistantOutcome:\n            append_settled((outcome.message,))\n            await emit(EventType.MESSAGE_END, attempt=max(total_attempts, 1))\n            await emit(EventType.AGENT_END, attempt=max(total_attempts, 1))\n            return outcome\n\n        try:\n            if cancellation.requested:\n                raise asyncio.CancelledError\n            await emit(EventType.AGENT_START)\n            try:\n                hooked_history = await self._apply_hooks(\n                    HookPoint.BEFORE_RUN,\n                    tuple(request_history),\n                    snapshot,\n                )\n                if isinstance(hooked_history, (str, bytes)) or not isinstance(\n                    hooked_history, Sequence\n                ):\n                    raise TypeError(\n                        "before_run Hook must return a message sequence or None"\n                    )\n                accepted_hooked_history = tuple(hooked_history)\n                to_model_messages(accepted_hooked_history)\n                request_history[:] = accepted_hooked_history\n            except (HookExecutionError, TypeError, ValueError) as hook_error:\n                error = ModelError(\n                    ModelErrorCode.HOOK_FAILED,\n                    str(hook_error),\n                    False,\n                )\n                return await finish(\n                    AssistantOutcome(\n                        AgentMessage.text(Role.ASSISTANT, ""),\n                        StopReason.ERROR,\n                        run_usage,\n                        error,\n                        1,\n                        tuple(run_tool_results),\n                        TerminalStatus.MODEL_ERROR,\n                    )\n                )\n            while True:\n                turn_attempt = 0\n                while True:\n                    turn_attempt += 1\n                    total_attempts += 1\n                    attempt = total_attempts\n                    request = ModelRequest(\n                        to_model_messages(request_history),\n                        snapshot.model,\n                        tuple(tool.definition() for tool in tools.values()),\n                    )\n                    try:\n                        hooked_request = await self._apply_hooks(\n                            HookPoint.BEFORE_MODEL_REQUEST, request, snapshot\n                        )\n                        if not isinstance(hooked_request, ModelRequest):\n                            raise TypeError(\n                                "before_model_request Hook must return ModelRequest or None"\n                            )\n                        request = hooked_request\n                    except (HookExecutionError, TypeError) as hook_error:\n                        error = ModelError(\n                            ModelErrorCode.HOOK_FAILED,\n                            str(hook_error),\n                            False,\n                        )\n                        return await finish(\n                            AssistantOutcome(\n                                AgentMessage.text(Role.ASSISTANT, ""),\n                                StopReason.ERROR,\n                                run_usage,\n                                error,\n                                max(total_attempts, 1),\n                                tuple(run_tool_results),\n                                TerminalStatus.MODEL_ERROR,\n                            )\n                        )\n                    await emit(EventType.MODEL_ATTEMPT_START, attempt=attempt)\n                    text_parts = []\n                    tool_drafts: dict[int, dict[str, str]] = {}\n                    current_usage = None\n                    end: ModelEnd | None = None\n                    schema_error: ModelError | None = None\n                    try:\n                        async for event in snapshot.adapter.stream(request):\n                            await emit(\n                                EventType.MODEL_EVENT,\n                                attempt=attempt,\n                                model_event=event,\n                            )\n                            if end is not None:\n                                schema_error = ModelError(\n                                    ModelErrorCode.SCHEMA,\n                                    "model stream emitted data after ModelEnd",\n                                    False,\n                                )\n                                break\n                            if isinstance(event, TextDelta):\n                                text_parts.append(event.text)\n                            elif isinstance(event, ToolCallDelta):\n                                if event.index < 0:\n                                    schema_error = ModelError(\n                                        ModelErrorCode.SCHEMA,\n                                        "model stream emitted an invalid Tool Call index",\n                                        False,\n                                    )\n                                    break\n                                draft = tool_drafts.setdefault(\n                                    event.index,\n                                    {"id": "", "name": "", "arguments": ""},\n                                )\n                                draft["id"] += event.id\n                                draft["name"] += event.name\n                                draft["arguments"] += event.arguments_delta\n                            elif isinstance(event, UsageUpdate):\n                                current_usage = event.usage\n                            elif isinstance(event, ModelEnd):\n                                end = event\n                            else:\n                                schema_error = ModelError(\n                                    ModelErrorCode.SCHEMA,\n                                    "model stream emitted an unsupported event",\n                                    False,\n                                )\n                                break\n                    except ModelAdapterError as failure:\n                        partial_text = "".join(text_parts)\n                        await emit(\n                            EventType.MODEL_ATTEMPT_FAILED,\n                            attempt=attempt,\n                            error=failure.error,\n                            partial_text=partial_text,\n                            partial_usage=current_usage,\n                        )\n                        if (\n                            failure.error.code is ModelErrorCode.CONTEXT_OVERFLOW\n                            and not overflow_recovery_attempted\n                            and self._context_overflow_recovery is not None\n                        ):\n                            overflow_recovery_attempted = True\n                            await emit(\n                                EventType.COMPACTION_START,\n                                attempt=attempt,\n                                error=failure.error,\n                                operation=ModelOperation.COMPACTION,\n                            )\n                            try:\n                                recovered = await self._context_overflow_recovery()\n                            except Exception as recovery_error:\n                                recovered = False\n                                failure = ModelAdapterError(\n                                    ModelError(\n                                        ModelErrorCode.COMPACTION_FAILED,\n                                        "context overflow recovery Compaction failed: "\n                                        f"{type(recovery_error).__name__}",\n                                        False,\n                                    )\n                                )\n                            await emit(\n                                (\n                                    EventType.COMPACTION_END\n                                    if recovered\n                                    else EventType.COMPACTION_FAILED\n                                ),\n                                attempt=attempt,\n                                error=None if recovered else failure.error,\n                                operation=ModelOperation.COMPACTION,\n                            )\n                            if recovered:\n                                request_history[:] = self.effective_history\n                                text_parts = []\n                                current_usage = None\n                                continue\n                        delay = snapshot.retry_policy.delay_for(\n                            failure.error,\n                            turn_attempt,\n                            retry_after_seconds=failure.error.retry_after_seconds,\n                        )\n                        if delay is not None:\n                            await emit(\n                                EventType.RETRY_SCHEDULED,\n                                attempt=attempt,\n                                error=failure.error,\n                                retry_delay_seconds=delay,\n                                partial_text=partial_text,\n                                partial_usage=current_usage,\n                            )\n                            text_parts = []\n                            current_usage = None\n                            await snapshot.sleeper(delay)\n                            continue\n                        terminal_usage = _add_usage(run_usage, current_usage)\n                        return await finish(\n                            AssistantOutcome(\n                                AgentMessage.text(Role.ASSISTANT, partial_text),\n                                StopReason.ERROR,\n                                terminal_usage,\n                                failure.error,\n                                total_attempts,\n                                tuple(run_tool_results),\n                                TerminalStatus.MODEL_ERROR,\n                            )\n                        )\n\n                    stream_error = schema_error\n                    if stream_error is None and end is None:\n                        stream_error = ModelError(\n                            ModelErrorCode.SCHEMA,\n                            "model stream violated the provider-neutral event contract",\n                            False,\n                        )\n                    blocks: list[ContentBlock] = []\n                    if stream_error is None:\n                        try:\n                            if text_parts:\n                                blocks.append(TextContent("".join(text_parts)))\n                            for index in sorted(tool_drafts):\n                                blocks.append(ToolCallContent(**tool_drafts[index]))\n                        except (TypeError, ValueError):\n                            stream_error = ModelError(\n                                ModelErrorCode.SCHEMA,\n                                "model stream emitted an incomplete Tool Call",\n                                False,\n                            )\n                    if stream_error is not None:\n                        partial_text = "".join(text_parts)\n                        await emit(\n                            EventType.MODEL_ATTEMPT_FAILED,\n                            attempt=attempt,\n                            error=stream_error,\n                            partial_text=partial_text,\n                            partial_usage=current_usage,\n                        )\n                        terminal_usage = _add_usage(run_usage, current_usage)\n                        return await finish(\n                            AssistantOutcome(\n                                AgentMessage.text(Role.ASSISTANT, partial_text),\n                                StopReason.ERROR,\n                                terminal_usage,\n                                stream_error,\n                                total_attempts,\n                                tuple(run_tool_results),\n                                TerminalStatus.MODEL_ERROR,\n                            )\n                        )\n                    assert end is not None\n                    break\n\n                run_usage = _add_usage(run_usage, current_usage)\n                turns += 1\n                assistant = AgentMessage(Role.ASSISTANT, tuple(blocks))\n                calls = tuple(\n                    block\n                    for block in assistant.content\n                    if isinstance(block, ToolCallContent)\n                )\n                if calls:\n                    append_unsettled(assistant)\n                else:\n                    append_settled((assistant,))\n                await emit(EventType.MESSAGE_END, attempt=total_attempts)\n                if not calls:\n                    if self._settled_turn_handler is not None:\n                        previous_effective = self.effective_history\n                        await self._settled_turn_handler()\n                        if self.effective_history != previous_effective:\n                            request_history[:] = self.effective_history\n                    if turn_input is not None and turn_input.pending():\n                        reached = guard_status()\n                        if reached is not None:\n                            await emit(EventType.AGENT_END, attempt=total_attempts)\n                            return AssistantOutcome(\n                                assistant,\n                                StopReason.ABORTED,\n                                run_usage,\n                                attempts=total_attempts,\n                                tool_results=tuple(run_tool_results),\n                                status=reached,\n                            )\n                        steering = tuple(turn_input.take())\n                        to_model_messages(steering)\n                        append_settled(steering)\n                        continue\n                    await emit(EventType.AGENT_END, attempt=total_attempts)\n                    return AssistantOutcome(\n                        assistant,\n                        end.stop_reason,\n                        run_usage,\n                        attempts=total_attempts,\n                        tool_results=tuple(run_tool_results),\n                    )\n\n                await emit(EventType.TOOL_BATCH_START, attempt=total_attempts)\n                prepared: dict[int, PreparedToolCall] = {}\n                results: dict[int, ToolResult] = {}\n                for index, call in enumerate(calls):\n                    tool = tools.get(call.name)\n                    if tool is None:\n                        results[index] = ToolResult.error(\n                            ToolErrorCode.UNKNOWN_TOOL, call.name\n                        )\n                        continue\n                    try:\n                        parsed = json.loads(call.arguments)\n                    except (json.JSONDecodeError, TypeError):\n                        results[index] = ToolResult.error(\n                            ToolErrorCode.INVALID_JSON, call.name\n                        )\n                        continue\n                    try:\n                        Draft202012Validator(tool.input_schema).validate(parsed)\n                    except ValidationError:\n                        results[index] = ToolResult.error(\n                            ToolErrorCode.INVALID_ARGUMENTS, call.name\n                        )\n                        continue\n                    if not isinstance(parsed, dict):\n                        results[index] = ToolResult.error(\n                            ToolErrorCode.INVALID_ARGUMENTS, call.name\n                        )\n                        continue\n                    candidate = PreparedToolCall(call, tool, parsed)\n                    try:\n                        hooked_call = await self._apply_hooks(\n                            HookPoint.BEFORE_TOOL_CALL,\n                            candidate,\n                            snapshot,\n                        )\n                        if not isinstance(hooked_call, PreparedToolCall):\n                            raise TypeError(\n                                "before_tool_call Hook must return PreparedToolCall or None"\n                            )\n                    except (HookExecutionError, TypeError):\n                        results[index] = ToolResult.error(\n                            ToolErrorCode.HOOK_FAILED,\n                            call.name,\n                        )\n                        continue\n                    prepared[index] = hooked_call\n\n                async def execute_one(index: int, call: PreparedToolCall) -> None:\n                    await emit(\n                        EventType.TOOL_CALL_START,\n                        attempt=total_attempts,\n                        tool_call_id=call.call.id,\n                        tool_name=call.call.name,\n                        tool_arguments=call.arguments,\n                    )\n                    try:\n                        result = await snapshot.tool_executor.execute(call)\n                        if not isinstance(result, ToolResult):\n                            raise TypeError("ToolExecutor returned an invalid result")\n                    except Exception:\n                        result = ToolResult.error(\n                            ToolErrorCode.EXECUTION_FAILED, call.call.name\n                        )\n                    try:\n                        hooked_result = await self._apply_hooks(\n                            HookPoint.AFTER_TOOL_CALL,\n                            result,\n                            snapshot,\n                        )\n                        if not isinstance(hooked_result, ToolResult):\n                            raise TypeError(\n                                "after_tool_call Hook must return ToolResult or None"\n                            )\n                        result = hooked_result\n                    except (HookExecutionError, TypeError):\n                        result = ToolResult.error(\n                            ToolErrorCode.HOOK_FAILED,\n                            call.call.name,\n                        )\n                    result = bound_tool_result(\n                        result,\n                        snapshot.tool_output_budget,\n                        call.tool.output_direction,\n                    )\n                    results[index] = result\n                    await emit(\n                        EventType.TOOL_CALL_END,\n                        attempt=total_attempts,\n                        tool_call_id=call.call.id,\n                        tool_name=call.call.name,\n                        tool_result=result,\n                    )\n\n                try:\n                    if any(call.tool.sequential for call in prepared.values()):\n                        for index, prepared_call in prepared.items():\n                            await execute_one(index, prepared_call)\n                    else:\n                        tasks = {\n                            index: asyncio.create_task(\n                                execute_one(index, prepared_call)\n                            )\n                            for index, prepared_call in prepared.items()\n                        }\n                        try:\n                            await asyncio.gather(*tasks.values())\n                        except asyncio.CancelledError:\n                            for task in tasks.values():\n                                if not task.done():\n                                    task.cancel()\n                            await asyncio.gather(\n                                *tasks.values(), return_exceptions=True\n                            )\n                            raise\n                except asyncio.CancelledError:\n                    for index, prepared_call in prepared.items():\n                        if index not in results:\n                            cancelled_result = ToolResult.error(\n                                ToolErrorCode.CANCELLED,\n                                prepared_call.call.name,\n                            )\n                            results[index] = cancelled_result\n                            await emit(\n                                EventType.TOOL_CALL_END,\n                                attempt=total_attempts,\n                                tool_call_id=prepared_call.call.id,\n                                tool_name=prepared_call.call.name,\n                                tool_result=cancelled_result,\n                            )\n                    for index, call in enumerate(calls):\n                        result = results[index]\n                        run_tool_results.append(result)\n                        append_unsettled(\n                            ToolResultMessage(call.id, call.name, result)\n                        )\n                    self._mark_settled(self._history[-(len(calls) + 1) :])\n                    await emit(EventType.TOOL_BATCH_END, attempt=total_attempts)\n                    raise\n\n                batch_results: list[ToolResult] = []\n                for index, call in enumerate(calls):\n                    result = results[index]\n                    batch_results.append(result)\n                    run_tool_results.append(result)\n                    append_unsettled(\n                        ToolResultMessage(call.id, call.name, result)\n                    )\n                self._mark_settled(self._history[-(len(calls) + 1) :])\n                tool_calls += len(calls)\n                await emit(EventType.TOOL_BATCH_END, attempt=total_attempts)\n                if self._settled_turn_handler is not None:\n                    previous_effective = self.effective_history\n                    await self._settled_turn_handler()\n                    if self.effective_history != previous_effective:\n                        request_history[:] = self.effective_history\n                if batch_results and all(result.terminate for result in batch_results):\n                    await emit(EventType.AGENT_END, attempt=total_attempts)\n                    return AssistantOutcome(\n                        assistant,\n                        end.stop_reason,\n                        run_usage,\n                        attempts=total_attempts,\n                        tool_results=tuple(run_tool_results),\n                    )\n                reached = guard_status()\n                if reached is not None:\n                    await emit(EventType.AGENT_END, attempt=total_attempts)\n                    return AssistantOutcome(\n                        assistant,\n                        StopReason.ABORTED,\n                        run_usage,\n                        attempts=total_attempts,\n                        tool_results=tuple(run_tool_results),\n                        status=reached,\n                    )\n                if turn_input is not None and turn_input.pending():\n                    steering = tuple(turn_input.take())\n                    to_model_messages(steering)\n                    append_settled(steering)\n        except asyncio.CancelledError:\n            message = AgentMessage.text(Role.ASSISTANT, "".join(text_parts))\n            outcome = AssistantOutcome(\n                message,\n                StopReason.ABORTED,\n                _add_usage(run_usage, current_usage),\n                attempts=max(total_attempts, 1),\n                tool_results=tuple(run_tool_results),\n                status=cancellation.status,\n            )\n            append_settled((message,))\n            await emit(\n                EventType.RUN_CANCELLED,\n                attempt=max(total_attempts, 1),\n                partial_text="".join(text_parts),\n                partial_usage=current_usage,\n            )\n            await emit(EventType.MESSAGE_END, attempt=max(total_attempts, 1))\n            await emit(EventType.AGENT_END, attempt=max(total_attempts, 1))\n            return outcome\n        finally:\n            await events.put(_EVENTS_DONE)\n'

In [ ]:
SESSION_SOURCE = '"""Application-facing control for one in-memory agent conversation."""\n\nfrom __future__ import annotations\n\nimport asyncio\nfrom collections import deque\nfrom dataclasses import dataclass, replace\nfrom collections.abc import AsyncIterator, Mapping, Sequence\nfrom enum import Enum\nfrom typing import Protocol, cast\n\nfrom .compaction import (\n    CharacterTokenEstimator,\n    CompactionCheckpoint,\n    CompactionPolicy,\n    CompactionStrategy,\n    CompactionTrigger,\n    CompactionWarning,\n    CompactionWarningCode,\n    StructuredSummary,\n    TokenEstimator,\n)\nfrom .extensions import (\n    CompactionHookRequest,\n    CommandExecutionError,\n    Extension,\n    ExtensionHost,\n    HookPoint,\n    ReloadResult,\n)\nfrom .model import AgentMessage, Role, StopReason\nfrom .persistence import SessionBusyError, SessionStore, SessionWriter\nfrom .runtime import (\n    AgentRunHandle,\n    AgentRuntime,\n    AssistantOutcome,\n    RunSnapshot,\n    RuntimeEvent,\n)\n\n\n_SESSION_EVENTS_DONE = object()\n\n\nclass InputKind(str, Enum):\n    STEERING = "steering"\n    FOLLOW_UP = "follow_up"\n\n\n@dataclass(frozen=True, slots=True)\nclass PendingInput:\n    kind: InputKind\n    message: AgentMessage\n\n\n@dataclass(frozen=True, slots=True)\nclass SessionRunResult:\n    outcome: AssistantOutcome\n    outcomes: tuple[AssistantOutcome, ...] = ()\n    pending_inputs: tuple[PendingInput, ...] = ()\n    trace_complete: bool = True\n    trace_error: str | None = None\n\n\nclass TracePersistenceError(RuntimeError):\n    """Strict tracing could not retain the evidence for a Run."""\n\n\nclass RunObserver(Protocol):\n    """Replaceable observation seam used by durable Run stores."""\n\n    def start_run(self, session_id: str | None, snapshot: RunSnapshot) -> None: ...\n\n    def record_event(self, event: RuntimeEvent) -> None: ...\n\n    def finish_run(\n        self, session_id: str | None, run_id: str, result: SessionRunResult\n    ) -> None: ...\n\n    def mark_incomplete(self, run_id: str, reason: str) -> None: ...\n\n\n@dataclass(frozen=True, slots=True)\nclass CompactionResult:\n    checkpoint: CompactionCheckpoint\n\n\nclass SessionRunHandle:\n    def __init__(\n        self,\n        task: asyncio.Task[SessionRunResult],\n        session: "AgentSession",\n        events: asyncio.Queue[RuntimeEvent | object],\n        snapshot: RunSnapshot,\n    ) -> None:\n        self._task = task\n        self._session = session\n        self._events = events\n        self._snapshot = snapshot\n\n    @property\n    def run_id(self) -> str:\n        return self._snapshot.run_id\n\n    @property\n    def snapshot(self) -> RunSnapshot:\n        return self._snapshot\n\n    async def result(self) -> SessionRunResult:\n        return await self._task\n\n    def cancel(self) -> None:\n        if not self._task.done():\n            self._session.cancel()\n\n    async def events(self) -> AsyncIterator[RuntimeEvent]:\n        while True:\n            event = await self._events.get()\n            if event is _SESSION_EVENTS_DONE:\n                break\n            yield cast(RuntimeEvent, event)\n\n\nclass _SteeringQueue:\n    def __init__(self) -> None:\n        self._messages: deque[PendingInput] = deque()\n\n    def append(self, message: AgentMessage) -> None:\n        self._messages.append(PendingInput(InputKind.STEERING, message))\n\n    def pending(self) -> bool:\n        return bool(self._messages)\n\n    def take(self) -> Sequence[AgentMessage]:\n        return (self._messages.popleft().message,)\n\n    def drain(self) -> tuple[PendingInput, ...]:\n        drained = tuple(self._messages)\n        self._messages.clear()\n        return drained\n\n\nclass AgentSession:\n    """Coordinate one active Run with optional durable Session state."""\n\n    def __init__(\n        self,\n        runtime: AgentRuntime,\n        *,\n        store: SessionStore | None = None,\n        session_id: str | None = None,\n        parent_entry_id: str | None = None,\n        compaction_policy: CompactionPolicy | None = None,\n        compaction_strategy: CompactionStrategy | None = None,\n        token_estimator: TokenEstimator | None = None,\n        extensions: Sequence[Extension] = (),\n        prompt_hashes: Mapping[str, str] | None = None,\n        resource_hashes: Mapping[str, str] | None = None,\n        observer: RunObserver | None = None,\n        strict_tracing: bool = False,\n    ) -> None:\n        if not isinstance(runtime, AgentRuntime):\n            raise TypeError("runtime must be an AgentRuntime")\n        if (store is None) != (session_id is None):\n            raise ValueError("durable Sessions require both store and session_id")\n        if compaction_policy is not None and not isinstance(\n            compaction_policy, CompactionPolicy\n        ):\n            raise TypeError("compaction_policy must be a CompactionPolicy")\n        if compaction_strategy is not None and not callable(\n            getattr(compaction_strategy, "plan", None)\n        ):\n            raise TypeError("compaction_strategy must provide plan()")\n        if token_estimator is not None and not callable(\n            getattr(token_estimator, "estimate", None)\n        ):\n            raise TypeError("token_estimator must provide estimate()")\n        self._runtime = runtime\n        self._base_tools = runtime.tools\n        self._extension_host = ExtensionHost(self._base_tools, extensions)\n        runtime.configure_tools(self._extension_host.tools)\n        runtime.configure_subscribers(self._extension_host.subscribers)\n        runtime.configure_hooks(self._extension_host.hooks)\n        self._store = store\n        self._session_id = session_id\n        self._parent_entry_id = parent_entry_id\n        self._compaction_policy = compaction_policy or CompactionPolicy()\n        self._compaction_strategy = compaction_strategy or CompactionStrategy()\n        self._token_estimator = token_estimator or CharacterTokenEstimator()\n        self._prompt_hashes = dict(prompt_hashes or {})\n        self._resource_hashes = dict(resource_hashes or {})\n        self._observer = observer\n        self._strict_tracing = strict_tracing\n        self._trace_complete = True\n        self._trace_error: str | None = None\n        if any(\n            not isinstance(name, str)\n            or not name\n            or not isinstance(digest, str)\n            or not digest\n            for name, digest in (\n                *self._prompt_hashes.items(),\n                *self._resource_hashes.items(),\n            )\n        ):\n            raise ValueError("snapshot hashes require non-empty string names and values")\n        self._flush_pending_custom_entries(self._extension_host)\n        self._warnings: list[CompactionWarning] = []\n        self._busy = False\n        self._active: AgentRunHandle | None = None\n        self._compaction_task: asyncio.Task[object] | None = None\n        self._cancel_requested = False\n        self._steering = _SteeringQueue()\n        self._follow_ups: deque[PendingInput] = deque()\n\n    @property\n    def busy(self) -> bool:\n        return self._busy\n\n    @property\n    def session_id(self) -> str | None:\n        return self._session_id\n\n    @property\n    def warnings(self) -> tuple[CompactionWarning, ...]:\n        return tuple(self._warnings)\n\n    @property\n    def extension_replacements(self):\n        return self._extension_host.replacements\n\n    def run_annotations(self, run_id: str | None = None):\n        return self._extension_host.run_annotations(run_id)\n\n    def _flush_pending_custom_entries(self, host: ExtensionHost) -> None:\n        if self._store is None:\n            return\n        assert self._session_id is not None\n        with self._store.writer(self._session_id) as writer:\n            def persist_custom(custom) -> None:\n                entry = writer.append_custom(\n                    custom,\n                    parent_id=self._parent_entry_id,\n                )\n                self._parent_entry_id = entry.entry_id\n\n            host.activate_custom_entry_sink(persist_custom)\n            host.deactivate_custom_entry_sink()\n\n    def start(self, prompt: str | AgentMessage) -> SessionRunHandle:\n        if self._busy:\n            raise SessionBusyError(\n                self._session_id or "ephemeral",\n                "Session already has an active Run",\n            )\n        message = (\n            AgentMessage.text(Role.USER, prompt) if isinstance(prompt, str) else prompt\n        )\n        if not isinstance(message, AgentMessage) or message.role is not Role.USER:\n            raise TypeError("prompt must be text or a user AgentMessage")\n        writer: SessionWriter | None = None\n        if self._store is not None:\n            assert self._session_id is not None\n            writer = self._store.writer(self._session_id)\n            writer = writer.__enter__()\n        self._busy = True\n        self._cancel_requested = False\n        self._trace_complete = True\n        self._trace_error = None\n        events: asyncio.Queue[RuntimeEvent | object] = asyncio.Queue()\n        try:\n            snapshot = self._runtime.capture_snapshot(\n                extension_identities=self._extension_host.identities,\n                compaction_policy=self._compaction_policy,\n                prompt_hashes=self._prompt_hashes,\n                resource_hashes=self._resource_hashes,\n            )\n            self._observe(snapshot.run_id, "start_run", self._session_id, snapshot)\n            task = asyncio.get_running_loop().create_task(\n                self._drive(message, events, writer, snapshot)\n            )\n        except BaseException:\n            self._busy = False\n            if writer is not None:\n                writer.__exit__(None, None, None)\n            raise\n        return SessionRunHandle(task, self, events, snapshot)\n\n    async def run(self, prompt: str | AgentMessage) -> SessionRunResult:\n        return await self.start(prompt).result()\n\n    async def prompt(self, prompt: str | AgentMessage) -> SessionRunResult:\n        return await self.run(prompt)\n\n    async def execute_command(self, name: str, arguments: str = "") -> object:\n        """Execute one registered chat command without starting an Agent Run."""\n\n        if self._busy:\n            raise SessionBusyError(\n                self._session_id or "ephemeral",\n                "chat commands require an idle Session",\n            )\n        registration = self._extension_host.command(name)\n        try:\n            return await registration.handler(arguments)\n        except Exception as error:\n            raise CommandExecutionError(name, type(error).__name__) from error\n\n    async def reload_extensions(\n        self,\n        extensions: Sequence[Extension],\n    ) -> ReloadResult:\n        """Replace explicit Extensions only while the Session is idle."""\n\n        if self._busy:\n            raise SessionBusyError(\n                self._session_id or "ephemeral",\n                "Extension reload requires an idle Session",\n            )\n        self._busy = True\n        old_host = self._extension_host\n        try:\n            warnings = await old_host.teardown()\n            replacement = ExtensionHost(self._base_tools, extensions)\n            old_names = {extension.name for extension in old_host.extensions}\n            new_names = {extension.name for extension in replacement.extensions}\n            self._extension_host = replacement\n            self._runtime.configure_tools(replacement.tools)\n            self._runtime.configure_subscribers(replacement.subscribers)\n            self._runtime.configure_hooks(replacement.hooks)\n            self._flush_pending_custom_entries(replacement)\n            return ReloadResult(\n                tuple(sorted(old_names & new_names)),\n                warnings,\n                replacement.replacements,\n            )\n        finally:\n            self._busy = False\n\n    async def compact(self, focus: str | None = None) -> CompactionResult:\n        """Create and install a context checkpoint at a Settled Boundary."""\n\n        if focus is not None and not focus.strip():\n            raise ValueError("Compaction focus cannot be empty")\n        if self._busy:\n            raise SessionBusyError(\n                self._session_id or "ephemeral",\n                "Compaction requires an idle Session at a Settled Boundary",\n            )\n        writer: SessionWriter | None = None\n        if self._store is not None:\n            assert self._session_id is not None\n            writer = self._store.writer(self._session_id).__enter__()\n        self._busy = True\n        try:\n            snapshot = self._runtime.capture_snapshot(\n                extension_identities=self._extension_host.identities,\n                compaction_policy=self._compaction_policy,\n                prompt_hashes=self._prompt_hashes,\n                resource_hashes=self._resource_hashes,\n            )\n            return await self._compact(\n                CompactionTrigger.MANUAL,\n                focus=focus,\n                writer=writer,\n                snapshot=snapshot,\n            )\n        finally:\n            if writer is not None:\n                writer.__exit__(None, None, None)\n            self._busy = False\n\n    async def _compact(\n        self,\n        trigger: CompactionTrigger,\n        *,\n        focus: str | None,\n        writer: SessionWriter | None,\n        snapshot: RunSnapshot | None = None,\n    ) -> CompactionResult:\n        current = asyncio.current_task()\n        previous = self._compaction_task\n        self._compaction_task = cast(asyncio.Task[object] | None, current)\n        try:\n            hook_request = await self._runtime.apply_hooks(\n                HookPoint.BEFORE_COMPACTION,\n                CompactionHookRequest(trigger, focus),\n                snapshot=snapshot,\n            )\n            if not isinstance(hook_request, CompactionHookRequest):\n                raise TypeError(\n                    "before_compaction Hook must return CompactionHookRequest or None"\n                )\n            if hook_request.trigger is not trigger:\n                raise ValueError("before_compaction Hook cannot change its trigger")\n            focus = hook_request.focus\n            configured_policy = (\n                snapshot.compaction_policy\n                if snapshot is not None\n                and isinstance(snapshot.compaction_policy, CompactionPolicy)\n                else self._compaction_policy\n            )\n            policy = configured_policy.resolve(\n                self._runtime.model if snapshot is None else snapshot.model\n            )\n            plan = self._compaction_strategy.plan(\n                self._runtime.effective_history,\n                keep_recent_tokens=policy.keep_recent_tokens,\n                estimator=self._token_estimator,\n            )\n            generated = await self._runtime.generate_summary(\n                plan.source,\n                focus=focus,\n                snapshot=snapshot,\n            )\n            checkpoint = CompactionCheckpoint(\n                trigger=trigger,\n                summary=StructuredSummary(generated.text, focus=focus),\n                tokens_before=plan.tokens_before,\n                summary_usage=generated.usage,\n                retained_tail=plan.retained_tail,\n            )\n            if writer is not None:\n                entry = writer.append_compaction(\n                    checkpoint,\n                    parent_id=self._parent_entry_id,\n                )\n                self._parent_entry_id = entry.entry_id\n            self._runtime.install_compaction(checkpoint)\n            return CompactionResult(checkpoint)\n        finally:\n            self._compaction_task = previous\n\n    async def _maybe_compact_after_settlement(\n        self,\n        writer: SessionWriter | None,\n        snapshot: RunSnapshot | None = None,\n    ) -> CompactionResult | None:\n        configured_policy = (\n            snapshot.compaction_policy\n            if snapshot is not None\n            and isinstance(snapshot.compaction_policy, CompactionPolicy)\n            else self._compaction_policy\n        )\n        resolved = configured_policy.resolve(\n            self._runtime.model if snapshot is None else snapshot.model\n        )\n        threshold = resolved.threshold_tokens\n        if threshold is None:\n            if (\n                resolved.context_window is None\n                and not any(\n                    warning.code is CompactionWarningCode.CONTEXT_WINDOW_UNKNOWN\n                    for warning in self._warnings\n                )\n            ):\n                self._warnings.append(\n                    CompactionWarning(\n                        CompactionWarningCode.CONTEXT_WINDOW_UNKNOWN,\n                        "threshold Compaction disabled because ModelSpec has no "\n                        "context_window; normalized overflow recovery remains enabled",\n                    )\n                )\n            return None\n        current_tokens = self._token_estimator.estimate(\n            self._runtime.effective_history\n        )\n        if current_tokens <= threshold:\n            return None\n        try:\n            return await self._compact(\n                CompactionTrigger.THRESHOLD,\n                focus=None,\n                writer=writer,\n                snapshot=snapshot,\n            )\n        except Exception as error:\n            self._warnings.append(\n                CompactionWarning(\n                    CompactionWarningCode.THRESHOLD_FAILED,\n                    "threshold Compaction failed; Session history unchanged: "\n                    f"{type(error).__name__}",\n                )\n            )\n            return None\n\n    def steer(self, message: str | AgentMessage) -> None:\n        if not self._busy:\n            raise RuntimeError("Steering requires an active Run")\n        self._steering.append(self._user_message(message))\n\n    def follow_up(self, message: str | AgentMessage) -> None:\n        if not self._busy:\n            raise RuntimeError("Follow-up requires an active Run")\n        self._follow_ups.append(\n            PendingInput(InputKind.FOLLOW_UP, self._user_message(message))\n        )\n\n    def cancel(self) -> None:\n        self._cancel_requested = True\n        if self._compaction_task is not None and not self._compaction_task.done():\n            self._compaction_task.cancel()\n        if self._active is not None:\n            self._active.cancel()\n\n    async def _drive(\n        self,\n        message: AgentMessage,\n        events: asyncio.Queue[RuntimeEvent | object],\n        writer: SessionWriter | None,\n        snapshot: RunSnapshot,\n    ) -> SessionRunResult:\n        outcomes: list[AssistantOutcome] = []\n\n        if writer is not None:\n            def settle(messages) -> None:\n                entry = writer.append(messages, parent_id=self._parent_entry_id)\n                self._parent_entry_id = entry.entry_id\n\n            self._runtime.set_settlement_sink(settle)\n\n            def persist_custom(custom) -> None:\n                entry = writer.append_custom(\n                    custom,\n                    parent_id=self._parent_entry_id,\n                )\n                self._parent_entry_id = entry.entry_id\n\n            self._extension_host.activate_custom_entry_sink(persist_custom)\n\n        async def recover_context_overflow() -> bool:\n            await self._compact(\n                CompactionTrigger.OVERFLOW,\n                focus=None,\n                writer=writer,\n                snapshot=snapshot,\n            )\n            return True\n\n        self._runtime.set_context_overflow_recovery(recover_context_overflow)\n        self._runtime.set_settled_turn_handler(\n            lambda: self._maybe_compact_after_settlement(writer, snapshot)\n        )\n\n        forwarded_sequence = 0\n\n        async def forward(handle: AgentRunHandle) -> None:\n            nonlocal forwarded_sequence\n            async for event in handle.events():\n                forwarded_sequence += 1\n                forwarded = replace(event, sequence=forwarded_sequence)\n                self._observe(snapshot.run_id, "record_event", forwarded)\n                await events.put(forwarded)\n\n        try:\n            await self._extension_host.startup()\n            next_message = message\n            while True:\n                self._active = self._runtime.start(\n                    [next_message],\n                    turn_input=self._steering,\n                    snapshot=snapshot,\n                )\n                if self._cancel_requested:\n                    self._active.cancel()\n                    # Let the Runtime\'s coordinated cancellation callback run\n                    # before a synchronous scripted stream can settle first.\n                    await asyncio.sleep(0)\n                forwarding = asyncio.create_task(forward(self._active))\n                outcome = await self._active.result()\n                await forwarding\n                outcomes.append(outcome)\n                if outcome.stop_reason in {StopReason.ABORTED, StopReason.ERROR}:\n                    break\n                if not self._follow_ups:\n                    break\n                next_message = self._follow_ups.popleft().message\n            pending = self._steering.drain() + tuple(self._follow_ups)\n            self._follow_ups.clear()\n            result = SessionRunResult(\n                outcome,\n                tuple(outcomes),\n                pending,\n                self._trace_complete,\n                self._trace_error,\n            )\n            self._observe(\n                snapshot.run_id,\n                "finish_run",\n                self._session_id,\n                snapshot.run_id,\n                result,\n            )\n            return SessionRunResult(\n                outcome,\n                tuple(outcomes),\n                pending,\n                self._trace_complete,\n                self._trace_error,\n            )\n        except BaseException as error:\n            self._mark_trace_incomplete(snapshot.run_id, type(error).__name__)\n            raise\n        finally:\n            self._runtime.set_settlement_sink(None)\n            self._runtime.set_context_overflow_recovery(None)\n            self._runtime.set_settled_turn_handler(None)\n            self._extension_host.deactivate_custom_entry_sink()\n            if writer is not None:\n                writer.__exit__(None, None, None)\n            self._active = None\n            self._cancel_requested = False\n            self._busy = False\n            await events.put(_SESSION_EVENTS_DONE)\n\n    def _observe(self, run_id: str, method: str, *arguments: object) -> None:\n        if self._observer is None or not self._trace_complete:\n            return\n        try:\n            getattr(self._observer, method)(*arguments)\n        except Exception as error:\n            self._mark_trace_incomplete(run_id, type(error).__name__)\n            if self._strict_tracing:\n                raise TracePersistenceError(\n                    f"Run Trace persistence failed during {method}: "\n                    f"{type(error).__name__}"\n                ) from error\n\n    def _mark_trace_incomplete(self, run_id: str, reason: str) -> None:\n        self._trace_complete = False\n        self._trace_error = reason\n        if self._observer is not None:\n            try:\n                self._observer.mark_incomplete(run_id, reason)\n            except Exception:\n                pass\n\n    @staticmethod\n    def _user_message(message: str | AgentMessage) -> AgentMessage:\n        accepted = (\n            AgentMessage.text(Role.USER, message)\n            if isinstance(message, str)\n            else message\n        )\n        if not isinstance(accepted, AgentMessage) or accepted.role is not Role.USER:\n            raise TypeError("Session input must be text or a user AgentMessage")\n        return accepted\n\n\ndef create_agent_session(\n    runtime: AgentRuntime,\n    *,\n    store: SessionStore | None = None,\n    session_id: str | None = None,\n    fork_from: str | None = None,\n    no_save: bool = False,\n    compaction_policy: CompactionPolicy | None = None,\n    compaction_strategy: CompactionStrategy | None = None,\n    token_estimator: TokenEstimator | None = None,\n    extensions: Sequence[Extension] = (),\n    prompt_hashes: Mapping[str, str] | None = None,\n    resource_hashes: Mapping[str, str] | None = None,\n    observer: RunObserver | None = None,\n    strict_tracing: bool = False,\n) -> AgentSession:\n    """Create a new durable Session, explicitly continue/fork one, or opt out."""\n\n    if no_save:\n        if store is not None or session_id is not None or fork_from is not None:\n            raise ValueError("no_save cannot be combined with persistence or continuation")\n        return AgentSession(\n            runtime,\n            compaction_policy=compaction_policy,\n            compaction_strategy=compaction_strategy,\n            token_estimator=token_estimator,\n            extensions=extensions,\n            prompt_hashes=prompt_hashes,\n            resource_hashes=resource_hashes,\n            observer=observer,\n            strict_tracing=strict_tracing,\n        )\n    if store is None:\n        if session_id is not None or fork_from is not None:\n            raise ValueError("continuation requires a SessionStore")\n        return AgentSession(\n            runtime,\n            compaction_policy=compaction_policy,\n            compaction_strategy=compaction_strategy,\n            token_estimator=token_estimator,\n            extensions=extensions,\n            prompt_hashes=prompt_hashes,\n            resource_hashes=resource_hashes,\n            observer=observer,\n            strict_tracing=strict_tracing,\n        )\n    if runtime.history:\n        raise ValueError("a durable AgentSession requires a fresh AgentRuntime")\n    if session_id is None:\n        if fork_from is not None:\n            raise ValueError("fork_from requires an existing session_id")\n        state = store.create()\n        leaf = None\n    else:\n        state = store.read(session_id)\n        leaf = fork_from if fork_from is not None else state.active_leaf_id\n        runtime.restore_history(\n            state.history(leaf),\n            effective_history=state.effective_history(leaf),\n        )\n    return AgentSession(\n        runtime,\n        store=store,\n        session_id=state.session_id,\n        parent_entry_id=leaf,\n        compaction_policy=compaction_policy,\n        compaction_strategy=compaction_strategy,\n        token_estimator=token_estimator,\n        extensions=extensions,\n        prompt_hashes=prompt_hashes,\n        resource_hashes=resource_hashes,\n        observer=observer,\n        strict_tracing=strict_tracing,\n    )\n'

In [ ]:
OBSERVABILITY_SOURCE = '"""Local, versioned Run evidence for reusable AgentSession execution."""\n\nfrom __future__ import annotations\n\nfrom collections.abc import Mapping, Sequence\nfrom dataclasses import dataclass, fields, is_dataclass\nfrom datetime import datetime, timezone\nfrom enum import Enum\nfrom hashlib import sha256\nimport json\nimport os\nfrom pathlib import Path\nimport platform\nimport re\nimport shutil\nimport subprocess\nimport sys\nimport tempfile\nimport time\nfrom typing import Protocol\n\nfrom platformdirs import user_data_path\n\nfrom .coding import ArtifactStore\nfrom .model import ModelEnd, TextDelta, ToolCallDelta, UsageUpdate\nfrom .persistence import JSONLSessionStore, SchemaVersion\nfrom .runtime import EventType, RunSnapshot, RuntimeEvent\nfrom .session import SessionRunResult\n\n\nRUN_TRACE_SCHEMA_VERSION = SchemaVersion(1, 0)\nRUN_ANNOTATION_SCHEMA_VERSION = SchemaVersion(1, 0)\nARTIFACT_SCHEMA_VERSION = SchemaVersion(1, 0)\n_SAFE_ID = re.compile(r"[A-Za-z0-9][A-Za-z0-9._-]{0,127}\\Z")\n_SENSITIVE_KEY = re.compile(\n    r"(?:api[_-]?key|authorization|credential|password|passwd|secret|token|"\n    r"private[_-]?key)",\n    re.IGNORECASE,\n)\n_BOUNDED_PREVIEW_KEY = re.compile(\n    r"(?:text|content|message|arguments|diagnostic|reason|preview|output)",\n    re.IGNORECASE,\n)\n\n\ndef _utc_now() -> str:\n    return datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")\n\n\ndef _version(version: SchemaVersion) -> dict[str, int]:\n    return {"major": version.major, "minor": version.minor}\n\n\ndef _safe_id(value: str, kind: str) -> str:\n    if not _SAFE_ID.fullmatch(value):\n        raise ValueError(f"{kind} id must be a safe local identifier")\n    return value\n\n\ndef _atomic_text(path: Path, content: str) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    descriptor, name = tempfile.mkstemp(prefix=f".{path.name}.", dir=path.parent)\n    temporary = Path(name)\n    try:\n        with os.fdopen(descriptor, "w", encoding="utf-8") as stream:\n            stream.write(content)\n            stream.flush()\n            os.fsync(stream.fileno())\n        os.replace(temporary, path)\n    except BaseException:\n        temporary.unlink(missing_ok=True)\n        raise\n\n\n@dataclass(frozen=True, slots=True)\nclass WorkspaceIdentity:\n    canonical_path: str\n    value: str\n\n    @classmethod\n    def from_path(cls, workspace: str | Path) -> "WorkspaceIdentity":\n        canonical = Path(workspace).resolve()\n        if not canonical.is_dir():\n            raise ValueError("workspace must be an existing directory")\n        normalized = os.path.normcase(str(canonical))\n        digest = sha256(normalized.encode("utf-8")).hexdigest()[:24]\n        return cls(str(canonical), f"workspace-{digest}")\n\n\n@dataclass(frozen=True, slots=True)\nclass RunSummary:\n    run_id: str\n    session_id: str | None\n    status: str | None\n    started_at: str\n    finished_at: str | None\n    evidence_complete: bool\n\n\n@dataclass(frozen=True, slots=True)\nclass PrunePreview:\n    run_ids: tuple[str, ...]\n    bytes: int\n\n\nclass UnsupportedRunSchemaVersionError(ValueError):\n    def __init__(self, artifact: str, found_major: int, supported_major: int) -> None:\n        self.artifact = artifact\n        self.found_major = found_major\n        self.supported_major = supported_major\n        super().__init__(\n            f"{artifact} schema major {found_major} is unsupported; this reader "\n            f"supports major {supported_major}. Preserve the original and migrate "\n            "it with migrate_run_trace()."\n        )\n\n\nclass TraceRedactor(Protocol):\n    def redact(self, value: object) -> object: ...\n\n\nclass StandardTraceRedactor:\n    """Remove credential-shaped fields and bound persisted string previews."""\n\n    def __init__(\n        self,\n        *,\n        secrets: Sequence[str] = (),\n        preview_chars: int = 4096,\n    ) -> None:\n        if preview_chars <= 0:\n            raise ValueError("preview_chars must be positive")\n        self._secrets = tuple(secret for secret in secrets if secret)\n        self._preview_chars = preview_chars\n\n    def redact(self, value: object) -> object:\n        if isinstance(value, Mapping):\n            return {\n                str(key): (\n                    "[REDACTED]"\n                    if _SENSITIVE_KEY.search(str(key))\n                    else self._redact_text(\n                        item,\n                        bound=_BOUNDED_PREVIEW_KEY.search(str(key)) is not None,\n                    )\n                    if isinstance(item, str)\n                    else self.redact(item)\n                )\n                for key, item in value.items()\n            }\n        if isinstance(value, (list, tuple)):\n            return [self.redact(item) for item in value]\n        if isinstance(value, str):\n            return self._redact_text(value, bound=True)\n        if value is None or isinstance(value, (bool, int, float)):\n            return value\n        return str(value)\n\n    def redact_artifact_text(self, value: str) -> str:\n        """Redact secrets without applying the bounded Trace preview limit."""\n\n        return self._redact_text(value, bound=False)\n\n    def _redact_text(self, value: str, *, bound: bool) -> str:\n        sanitized = value\n        for secret in self._secrets:\n            sanitized = sanitized.replace(secret, "[REDACTED]")\n        if bound and len(sanitized) > self._preview_chars:\n            omitted = len(sanitized) - self._preview_chars\n            sanitized = f"{sanitized[: self._preview_chars]}…[{omitted} chars omitted]"\n        return sanitized\n\n\nclass RunStore(Protocol):\n    def list(\n        self, *, session_id: str | None = None, status: str | None = None\n    ) -> tuple[RunSummary, ...]: ...\n\n    def export(self, run_id: str) -> str: ...\n\n    def annotate(\n        self,\n        run_id: str,\n        namespace: str,\n        payload: Mapping[str, object],\n    ) -> None: ...\n\n\nclass JSONLRunStore:\n    """File-per-Run JSONL store with append-only annotations and artifacts."""\n\n    def __init__(\n        self,\n        root: str | Path,\n        *,\n        workspace_identity: WorkspaceIdentity | None = None,\n        redactor: TraceRedactor | None = None,\n    ) -> None:\n        self.root = Path(root).resolve()\n        self.runs_directory = self.root / "runs"\n        self.workspace_identity = workspace_identity\n        self.redactor = redactor or StandardTraceRedactor()\n\n    def directory_for(self, run_id: str) -> Path:\n        return self.runs_directory / _safe_id(run_id, "Run")\n\n    def trace_path(self, run_id: str) -> Path:\n        return self.directory_for(run_id) / "trace.jsonl"\n\n    def annotation_path(self, run_id: str) -> Path:\n        return self.directory_for(run_id) / "annotations.jsonl"\n\n    def start(\n        self,\n        run_id: str,\n        *,\n        session_id: str | None,\n        snapshot: Mapping[str, object],\n    ) -> None:\n        accepted = _safe_id(run_id, "Run")\n        directory = self.directory_for(accepted)\n        directory.mkdir(parents=True, exist_ok=False)\n        record = {\n            "schema": "agent_harness.run_trace",\n            "schema_version": _version(RUN_TRACE_SCHEMA_VERSION),\n            "record": "run_start",\n            "run_id": accepted,\n            "session_id": session_id,\n            "workspace_identity": (\n                None if self.workspace_identity is None else self.workspace_identity.value\n            ),\n            "timestamp": _utc_now(),\n            "snapshot": dict(snapshot),\n        }\n        _atomic_text(self.trace_path(accepted), self._encode(record))\n\n    def append(self, run_id: str, record: str, payload: Mapping[str, object]) -> None:\n        accepted = _safe_id(run_id, "Run")\n        path = self.trace_path(accepted)\n        if not path.is_file():\n            raise KeyError(f"unknown Run: {accepted}")\n        value = {\n            "schema": "agent_harness.run_trace",\n            "schema_version": _version(RUN_TRACE_SCHEMA_VERSION),\n            "record": record,\n            "run_id": accepted,\n            "timestamp": _utc_now(),\n            **dict(payload),\n        }\n        with path.open("a", encoding="utf-8") as stream:\n            stream.write(self._encode(value))\n            stream.flush()\n            os.fsync(stream.fileno())\n\n    def mark_incomplete(self, run_id: str, reason: str) -> None:\n        if self.trace_path(run_id).is_file():\n            self.append(run_id, "trace_incomplete", {"reason": reason})\n\n    def records(self, run_id: str) -> tuple[Mapping[str, object], ...]:\n        records, _ = self._read_jsonl(\n            self.trace_path(run_id),\n            schema="agent_harness.run_trace",\n            version=RUN_TRACE_SCHEMA_VERSION,\n            artifact="Run Trace",\n        )\n        return records\n\n    def list(\n        self, *, session_id: str | None = None, status: str | None = None\n    ) -> tuple[RunSummary, ...]:\n        if not self.runs_directory.is_dir():\n            return ()\n        summaries: list[RunSummary] = []\n        for path in sorted(self.runs_directory.iterdir()):\n            if not path.is_dir() or not self.trace_path(path.name).is_file():\n                continue\n            summary = self.summary(path.name)\n            if session_id is not None and summary.session_id != session_id:\n                continue\n            if status is not None and summary.status != status:\n                continue\n            summaries.append(summary)\n        return tuple(sorted(summaries, key=lambda item: item.started_at, reverse=True))\n\n    def summary(self, run_id: str) -> RunSummary:\n        records, final_line_complete = self._read_jsonl(\n            self.trace_path(run_id),\n            schema="agent_harness.run_trace",\n            version=RUN_TRACE_SCHEMA_VERSION,\n            artifact="Run Trace",\n        )\n        if not records:\n            raise ValueError("Run Trace has no committed header")\n        header = records[0]\n        end = next(\n            (record for record in reversed(records) if record.get("record") == "run_end"),\n            None,\n        )\n        explicitly_incomplete = any(\n            record.get("record") == "trace_incomplete" for record in records\n        )\n        return RunSummary(\n            str(header["run_id"]),\n            None if header.get("session_id") is None else str(header["session_id"]),\n            None if end is None else str(end.get("status")),\n            str(header["timestamp"]),\n            None if end is None else str(end.get("timestamp")),\n            final_line_complete and end is not None and not explicitly_incomplete,\n        )\n\n    def export(self, run_id: str) -> str:\n        records = self.records(run_id)\n        return "".join(\n            json.dumps(record, ensure_ascii=False, sort_keys=True, separators=(",", ":"))\n            + "\\n"\n            for record in records\n        )\n\n    def annotate(\n        self,\n        run_id: str,\n        namespace: str,\n        payload: Mapping[str, object],\n    ) -> None:\n        accepted = _safe_id(run_id, "Run")\n        self.summary(accepted)\n        if not re.fullmatch(r"[a-z][a-z0-9_.-]{0,127}", namespace):\n            raise ValueError("annotation namespace must be a safe dotted name")\n        path = self.annotation_path(accepted)\n        record = {\n            "schema": "agent_harness.run_annotation",\n            "schema_version": _version(RUN_ANNOTATION_SCHEMA_VERSION),\n            "record": "annotation",\n            "run_id": accepted,\n            "timestamp": _utc_now(),\n            "namespace": namespace,\n            "payload": dict(payload),\n        }\n        path.parent.mkdir(parents=True, exist_ok=True)\n        with path.open("a", encoding="utf-8") as stream:\n            stream.write(self._encode(record))\n            stream.flush()\n            os.fsync(stream.fileno())\n\n    def annotations(self, run_id: str) -> tuple[Mapping[str, object], ...]:\n        path = self.annotation_path(run_id)\n        if not path.exists():\n            return ()\n        records, _ = self._read_jsonl(\n            path,\n            schema="agent_harness.run_annotation",\n            version=RUN_ANNOTATION_SCHEMA_VERSION,\n            artifact="Run Annotation",\n        )\n        return records\n\n    def put_artifact(\n        self,\n        run_id: str,\n        content: bytes,\n        *,\n        media_type: str = "text/plain; charset=utf-8",\n    ) -> str:\n        accepted = _safe_id(run_id, "Run")\n        self.summary(accepted)\n        digest = sha256(content).hexdigest()\n        path = self.directory_for(accepted) / "artifacts" / "sha256" / digest\n        if not path.exists():\n            path.parent.mkdir(parents=True, exist_ok=True)\n            descriptor, name = tempfile.mkstemp(prefix=".artifact.", dir=path.parent)\n            temporary = Path(name)\n            try:\n                with os.fdopen(descriptor, "wb") as stream:\n                    stream.write(content)\n                    stream.flush()\n                    os.fsync(stream.fileno())\n                os.replace(temporary, path)\n            except BaseException:\n                temporary.unlink(missing_ok=True)\n                raise\n        reference = f"sha256:{digest}"\n        self.append(\n            accepted,\n            "artifact",\n            {\n                "reference": reference,\n                "schema_version": _version(ARTIFACT_SCHEMA_VERSION),\n                "media_type": media_type,\n                "bytes": len(content),\n            },\n        )\n        return reference\n\n    def prune_preview(\n        self, *, status: str | None = None, session_id: str | None = None\n    ) -> PrunePreview:\n        run_ids = tuple(\n            summary.run_id for summary in self.list(status=status, session_id=session_id)\n        )\n        total = sum(\n            path.stat().st_size\n            for run_id in run_ids\n            for path in self.directory_for(run_id).rglob("*")\n            if path.is_file()\n        )\n        return PrunePreview(run_ids, total)\n\n    def prune(self, preview: PrunePreview) -> None:\n        for run_id in preview.run_ids:\n            directory = self.directory_for(run_id)\n            if directory.parent != self.runs_directory:\n                raise ValueError("Run prune target escaped the Runs directory")\n            shutil.rmtree(directory)\n\n    def _encode(self, value: Mapping[str, object]) -> str:\n        redacted = self.redactor.redact(value)\n        return (\n            json.dumps(\n                redacted,\n                ensure_ascii=False,\n                allow_nan=False,\n                sort_keys=True,\n                separators=(",", ":"),\n            )\n            + "\\n"\n        )\n\n    @staticmethod\n    def _read_jsonl(\n        path: Path,\n        *,\n        schema: str,\n        version: SchemaVersion,\n        artifact: str,\n    ) -> tuple[tuple[Mapping[str, object], ...], bool]:\n        try:\n            content = path.read_bytes()\n        except FileNotFoundError:\n            raise KeyError(f"unknown {artifact}: {path.parent.name}") from None\n        lines = content.splitlines(keepends=True)\n        final_line_complete = not lines or lines[-1].endswith(b"\\n")\n        if not final_line_complete:\n            lines = lines[:-1]\n        decoded: list[Mapping[str, object]] = []\n        for line_number, raw in enumerate(lines, 1):\n            try:\n                value = json.loads(raw.decode("utf-8"))\n            except (UnicodeDecodeError, json.JSONDecodeError) as error:\n                raise ValueError(\n                    f"invalid {artifact} JSONL at line {line_number}"\n                ) from error\n            if not isinstance(value, dict) or value.get("schema") != schema:\n                raise ValueError(f"invalid {artifact} schema at line {line_number}")\n            raw_version = value.get("schema_version")\n            if not isinstance(raw_version, dict):\n                raise ValueError(f"invalid {artifact} version at line {line_number}")\n            major = raw_version.get("major")\n            if isinstance(major, bool) or not isinstance(major, int):\n                raise ValueError(f"invalid {artifact} version at line {line_number}")\n            if major != version.major:\n                raise UnsupportedRunSchemaVersionError(artifact, major, version.major)\n            decoded.append(value)\n        return tuple(decoded), final_line_complete\n\n\n@dataclass(frozen=True, slots=True)\nclass LocalWorkspace:\n    identity: WorkspaceIdentity\n    root: Path\n    sessions: JSONLSessionStore\n    runs: JSONLRunStore\n\n    @classmethod\n    def open(\n        cls,\n        workspace: str | Path,\n        *,\n        storage_root: str | Path | None = None,\n        redactor: TraceRedactor | None = None,\n    ) -> "LocalWorkspace":\n        identity = WorkspaceIdentity.from_path(workspace)\n        base = (\n            Path(storage_root).resolve()\n            if storage_root is not None\n            else user_data_path("omega", "agent-harness").resolve()\n        )\n        root = base / "workspaces" / identity.value\n        root.mkdir(parents=True, exist_ok=True)\n        metadata = root / "workspace.json"\n        if not metadata.exists():\n            _atomic_text(\n                metadata,\n                json.dumps(\n                    {\n                        "schema": "agent_harness.workspace",\n                        "schema_version": {"major": 1, "minor": 0},\n                        "identity": identity.value,\n                        "canonical_path": identity.canonical_path,\n                    },\n                    ensure_ascii=False,\n                    indent=2,\n                    sort_keys=True,\n                )\n                + "\\n",\n            )\n        return cls(\n            identity,\n            root,\n            JSONLSessionStore(root),\n            JSONLRunStore(root, workspace_identity=identity, redactor=redactor),\n        )\n\n\nclass RunArtifactStore(ArtifactStore):\n    """Route Coding Tool full output to the currently active durable Run."""\n\n    def __init__(self, store: JSONLRunStore) -> None:\n        self._store = store\n        self._run_id: str | None = None\n\n    def activate(self, run_id: str) -> None:\n        self._run_id = run_id\n\n    def deactivate(self, run_id: str) -> None:\n        if self._run_id == run_id:\n            self._run_id = None\n\n    def put_text(self, content: str) -> str:\n        if self._run_id is None:\n            raise RuntimeError("Run Artifact storage requires an active Run")\n        artifact_redactor = getattr(self._store.redactor, "redact_artifact_text", None)\n        sanitized = (\n            artifact_redactor(content)\n            if callable(artifact_redactor)\n            else self._store.redactor.redact(content)\n        )\n        if not isinstance(sanitized, str):\n            raise TypeError("Trace redactor must return text for text artifacts")\n        return self._store.put_artifact(\n            self._run_id, sanitized.encode("utf-8"), media_type="text/plain; charset=utf-8"\n        )\n\n\nclass StandardTraceSink:\n    """Translate Runtime Events into bounded, sanitized, queryable Run records."""\n\n    def __init__(\n        self,\n        store: JSONLRunStore,\n        *,\n        workspace: str | Path,\n        artifacts: RunArtifactStore | None = None,\n    ) -> None:\n        self.store = store\n        self.workspace = Path(workspace).resolve()\n        self.artifacts = artifacts\n        self._started: dict[str, float] = {}\n        self._first_token: set[str] = set()\n\n    def start_run(self, session_id: str | None, snapshot: RunSnapshot) -> None:\n        snapshot_record: dict[str, object] = {\n            "fingerprint": snapshot.fingerprint,\n            "model": {\n                "id": snapshot.model.model_id,\n                "context_window": snapshot.model.context_window,\n                "max_output_tokens": snapshot.model.max_output_tokens,\n                "supports_tools": snapshot.model.supports_tools,\n            },\n            "tools": [tool.name for tool in snapshot.tools],\n            "extensions": list(snapshot.extension_identities),\n            "prompt_hashes": dict(snapshot.prompt_hashes),\n            "resource_hashes": dict(snapshot.resource_hashes),\n            "retry_policy": {\n                "delays": list(snapshot.retry_policy.delays),\n                "max_retry_after_seconds": snapshot.retry_policy.max_retry_after_seconds,\n            },\n            "platform": {\n                "python": sys.version.split()[0],\n                "system": platform.system(),\n                "release": platform.release(),\n                "machine": platform.machine(),\n            },\n            "git": self._git_evidence(),\n        }\n        self.store.start(\n            snapshot.run_id,\n            session_id=session_id,\n            snapshot=snapshot_record,\n        )\n        self._started[snapshot.run_id] = time.perf_counter()\n        if self.artifacts is not None:\n            self.artifacts.activate(snapshot.run_id)\n\n    def record_event(self, event: RuntimeEvent) -> None:\n        if event.run_id is None:\n            raise ValueError("Runtime Event is missing its Run id")\n        elapsed = time.perf_counter() - self._started[event.run_id]\n        if (\n            event.type is EventType.MODEL_EVENT\n            and isinstance(event.model_event, TextDelta)\n            and event.model_event.text\n            and event.run_id not in self._first_token\n        ):\n            self._first_token.add(event.run_id)\n            self.store.append(\n                event.run_id,\n                "first_token",\n                {"latency_seconds": round(elapsed, 6)},\n            )\n        self.store.append(\n            event.run_id,\n            "event",\n            {\n                "sequence": event.sequence,\n                "type": event.type.value,\n                "attempt": event.attempt,\n                "elapsed_seconds": round(elapsed, 6),\n                "operation": event.operation.value,\n                "retry_delay_seconds": event.retry_delay_seconds,\n                "partial_text": event.partial_text,\n                "error": self._json_value(event.error),\n                "model_event": self._model_event(event.model_event),\n                "tool_call_id": event.tool_call_id,\n                "tool_name": event.tool_name,\n                "tool_arguments": event.tool_arguments,\n                "tool_result": self._json_value(event.tool_result),\n                "snapshot_fingerprint": event.snapshot_fingerprint,\n            },\n        )\n\n    def finish_run(\n        self,\n        session_id: str | None,\n        run_id: str,\n        result: SessionRunResult,\n    ) -> None:\n        outcome = result.outcome\n        usage = outcome.usage\n        self.store.append(\n            run_id,\n            "run_end",\n            {\n                "session_id": session_id,\n                "status": outcome.status.value,\n                "stop_reason": outcome.stop_reason.value,\n                "attempts": outcome.attempts,\n                "duration_seconds": round(\n                    time.perf_counter() - self._started[run_id], 6\n                ),\n                "usage": self._json_value(usage),\n                "usage_provenance": (\n                    None if usage is None else "estimated" if usage.estimated else "provider"\n                ),\n                "error": self._json_value(outcome.error),\n                "pending_inputs": len(result.pending_inputs),\n            },\n        )\n        if self.artifacts is not None:\n            self.artifacts.deactivate(run_id)\n        self._started.pop(run_id, None)\n\n    def mark_incomplete(self, run_id: str, reason: str) -> None:\n        self.store.mark_incomplete(run_id, reason)\n        if self.artifacts is not None:\n            self.artifacts.deactivate(run_id)\n        self._started.pop(run_id, None)\n\n    @staticmethod\n    def _json_value(value: object) -> object:\n        if value is None:\n            return None\n        if isinstance(value, Enum):\n            return value.value\n        if is_dataclass(value):\n            return {\n                field.name: StandardTraceSink._json_value(getattr(value, field.name))\n                for field in fields(value)\n            }\n        if isinstance(value, Mapping):\n            return {\n                str(key): StandardTraceSink._json_value(item)\n                for key, item in value.items()\n            }\n        if isinstance(value, (list, tuple)):\n            return [StandardTraceSink._json_value(item) for item in value]\n        return value\n\n    @classmethod\n    def _model_event(cls, event: object) -> object:\n        if event is None:\n            return None\n        value = cls._json_value(event)\n        if isinstance(event, TextDelta):\n            kind = "text_delta"\n        elif isinstance(event, ToolCallDelta):\n            kind = "tool_call_delta"\n        elif isinstance(event, UsageUpdate):\n            kind = "usage"\n        elif isinstance(event, ModelEnd):\n            kind = "model_end"\n        else:\n            kind = type(event).__name__\n        return {"type": kind, "value": value}\n\n    def _git_evidence(self) -> Mapping[str, object] | None:\n        try:\n            commit = subprocess.run(\n                ["git", "rev-parse", "HEAD"],\n                cwd=self.workspace,\n                check=True,\n                capture_output=True,\n                text=True,\n                timeout=2,\n            ).stdout.strip()\n            dirty = bool(\n                subprocess.run(\n                    ["git", "status", "--porcelain", "--untracked-files=no"],\n                    cwd=self.workspace,\n                    check=True,\n                    capture_output=True,\n                    text=True,\n                    timeout=2,\n                ).stdout.strip()\n            )\n        except (FileNotFoundError, subprocess.SubprocessError):\n            return None\n        return {"commit": commit, "dirty": dirty}\n\n\ndef migrate_run_trace(source: str | Path, destination: str | Path) -> Path:\n    """Copy a supported Run Trace to a new validated file, preserving source."""\n\n    source_path = Path(source).resolve()\n    destination_path = Path(destination).resolve()\n    if destination_path.exists():\n        raise FileExistsError(f"migration destination exists: {destination_path}")\n    records, complete = JSONLRunStore._read_jsonl(\n        source_path,\n        schema="agent_harness.run_trace",\n        version=RUN_TRACE_SCHEMA_VERSION,\n        artifact="Run Trace",\n    )\n    if not complete:\n        raise ValueError("cannot migrate an incomplete final Run Trace record")\n    encoded = "".join(\n        json.dumps(record, ensure_ascii=False, sort_keys=True, separators=(",", ":"))\n        + "\\n"\n        for record in records\n    )\n    _atomic_text(destination_path, encoded)\n    return destination_path\n'

In [ ]:
API_SOURCE = '"""Concise public factories over explicit, replaceable harness infrastructure."""\n\nfrom __future__ import annotations\n\nimport asyncio\nfrom collections.abc import Awaitable, Callable, Mapping, Sequence\nfrom pathlib import Path\n\nfrom .coding import CodingToolPreset, create_coding_tool_preset\nfrom .compaction import CompactionPolicy, CompactionStrategy, TokenEstimator\nfrom .extensions import Extension\nfrom .model import (\n    AgentMessage,\n    ModelAdapter,\n    ModelSpec,\n    OpenAICompatibleAdapter,\n    OpenAICompatibleConfig,\n    Role,\n)\nfrom .observability import (\n    JSONLRunStore,\n    LocalWorkspace,\n    RunArtifactStore,\n    StandardTraceRedactor,\n    StandardTraceSink,\n)\nfrom .persistence import SessionStore\nfrom .resources import PromptAssembler, ResourceLoader\nfrom .runtime import AgentRuntime, RetryPolicy, RunGuard\nfrom .session import AgentSession, RunObserver, create_agent_session\nfrom .tools import Tool, ToolExecutor, ToolOutputBudget\n\n\nSleeper = Callable[[float], Awaitable[None]]\n\n\ndef _adapter(\n    adapter_or_config: ModelAdapter | OpenAICompatibleConfig,\n) -> tuple[ModelAdapter, tuple[str, ...]]:\n    if isinstance(adapter_or_config, OpenAICompatibleConfig):\n        return (\n            OpenAICompatibleAdapter(adapter_or_config),\n            (adapter_or_config.api_key, *adapter_or_config.headers.values()),\n        )\n    if not callable(getattr(adapter_or_config, "stream", None)):\n        raise TypeError(\n            "adapter_or_config must be a ModelAdapter or OpenAICompatibleConfig"\n        )\n    return adapter_or_config, ()\n\n\ndef _local_observation(\n    workspace: Path,\n    *,\n    storage_root: str | Path | None,\n    trace_secrets: Sequence[str],\n) -> tuple[LocalWorkspace, StandardTraceSink, RunArtifactStore]:\n    redactor = StandardTraceRedactor(secrets=trace_secrets)\n    local = LocalWorkspace.open(\n        workspace,\n        storage_root=storage_root,\n        redactor=redactor,\n    )\n    artifacts = RunArtifactStore(local.runs)\n    return local, StandardTraceSink(\n        local.runs,\n        workspace=workspace,\n        artifacts=artifacts,\n    ), artifacts\n\n\ndef create_session(\n    adapter_or_config: ModelAdapter | OpenAICompatibleConfig,\n    model: ModelSpec,\n    *,\n    workspace: str | Path,\n    storage_root: str | Path | None = None,\n    session_store: SessionStore | None = None,\n    run_store: JSONLRunStore | None = None,\n    observer: RunObserver | None = None,\n    session_id: str | None = None,\n    fork_from: str | None = None,\n    no_save: bool = False,\n    strict_tracing: bool = False,\n    trace_secrets: Sequence[str] = (),\n    tools: Sequence[Tool] = (),\n    tool_executor: ToolExecutor | None = None,\n    tool_output_budget: ToolOutputBudget | None = None,\n    retry_policy: RetryPolicy | None = None,\n    sleeper: Sleeper = asyncio.sleep,\n    run_guard: RunGuard | None = None,\n    generation_settings: Mapping[str, object] | None = None,\n    compaction_policy: CompactionPolicy | None = None,\n    compaction_strategy: CompactionStrategy | None = None,\n    token_estimator: TokenEstimator | None = None,\n    extensions: Sequence[Extension] = (),\n) -> AgentSession:\n    """Create a general durable Session, or an explicit all-memory Session."""\n\n    workspace_path = Path(workspace).resolve()\n    if not workspace_path.is_dir():\n        raise ValueError("workspace must be an existing directory")\n    if not isinstance(model, ModelSpec):\n        raise TypeError("model must be a ModelSpec")\n    adapter, config_secrets = _adapter(adapter_or_config)\n    if no_save and any(\n        value is not None for value in (session_store, run_store, observer, session_id)\n    ):\n        raise ValueError("no_save disables Session, Trace, Annotation, and Artifact stores")\n    effective_store = session_store\n    effective_observer = observer\n    if not no_save:\n        if effective_store is None and run_store is None and effective_observer is None:\n            local, effective_observer, _ = _local_observation(\n                workspace_path,\n                storage_root=storage_root,\n                trace_secrets=(*config_secrets, *trace_secrets),\n            )\n            effective_store = local.sessions\n        elif effective_observer is None and run_store is not None:\n            effective_observer = StandardTraceSink(\n                run_store,\n                workspace=workspace_path,\n            )\n    runtime = AgentRuntime(\n        adapter,\n        model,\n        tools=tools,\n        tool_executor=tool_executor,\n        tool_output_budget=tool_output_budget,\n        retry_policy=retry_policy,\n        sleeper=sleeper,\n        run_guard=run_guard,\n        generation_settings=generation_settings,\n    )\n    return create_agent_session(\n        runtime,\n        store=effective_store,\n        session_id=session_id,\n        fork_from=fork_from,\n        no_save=no_save,\n        compaction_policy=compaction_policy,\n        compaction_strategy=compaction_strategy,\n        token_estimator=token_estimator,\n        extensions=extensions,\n        observer=effective_observer,\n        strict_tracing=strict_tracing,\n    )\n\n\ndef create_coding_session(\n    adapter_or_config: ModelAdapter | OpenAICompatibleConfig,\n    model: ModelSpec,\n    *,\n    workspace: str | Path,\n    storage_root: str | Path | None = None,\n    session_store: SessionStore | None = None,\n    run_store: JSONLRunStore | None = None,\n    observer: RunObserver | None = None,\n    session_id: str | None = None,\n    fork_from: str | None = None,\n    no_save: bool = False,\n    strict_tracing: bool = False,\n    trace_secrets: Sequence[str] = (),\n    resource_loader: ResourceLoader | None = None,\n    prompt_assembler: PromptAssembler | None = None,\n    active_skills: Sequence[str] = (),\n    preset: CodingToolPreset | None = None,\n    retry_policy: RetryPolicy | None = None,\n    sleeper: Sleeper = asyncio.sleep,\n    run_guard: RunGuard | None = None,\n    generation_settings: Mapping[str, object] | None = None,\n    compaction_policy: CompactionPolicy | None = None,\n    compaction_strategy: CompactionStrategy | None = None,\n    token_estimator: TokenEstimator | None = None,\n    extensions: Sequence[Extension] = (),\n) -> AgentSession:\n    """Create the explicit Coding Agent preset over the same AgentSession seam."""\n\n    workspace_path = Path(workspace).resolve()\n    if not workspace_path.is_dir():\n        raise ValueError("workspace must be an existing directory")\n    adapter, config_secrets = _adapter(adapter_or_config)\n    if no_save and any(\n        value is not None for value in (session_store, run_store, observer, session_id)\n    ):\n        raise ValueError("no_save disables Session, Trace, Annotation, and Artifact stores")\n    effective_store = session_store\n    effective_observer = observer\n    artifact_store: RunArtifactStore | None = None\n    if not no_save:\n        if effective_store is None and run_store is None and effective_observer is None:\n            local, effective_observer, artifact_store = _local_observation(\n                workspace_path,\n                storage_root=storage_root,\n                trace_secrets=(*config_secrets, *trace_secrets),\n            )\n            effective_store = local.sessions\n        elif effective_observer is None and run_store is not None:\n            artifact_store = RunArtifactStore(run_store)\n            effective_observer = StandardTraceSink(\n                run_store,\n                workspace=workspace_path,\n                artifacts=artifact_store,\n            )\n    effective_preset = preset or create_coding_tool_preset(\n        workspace_path,\n        artifact_store=artifact_store,\n    )\n    if effective_preset.workspace.root != workspace_path:\n        raise ValueError("Coding Tool Preset must use the Session workspace")\n    loader = resource_loader or ResourceLoader(workspace_path)\n    if loader.workspace != workspace_path:\n        raise ValueError("ResourceLoader must use the Session workspace")\n    resources = loader.load()\n    assembler = prompt_assembler or PromptAssembler(\n        "You are Omega, a coding agent. Work carefully inside the supplied "\n        "workspace, use Tools for evidence, and verify changes before finishing."\n    )\n    prompt = assembler.assemble(\n        tools=effective_preset.tools,\n        resources=resources,\n        active_skills=active_skills,\n    )\n    runtime = AgentRuntime(\n        adapter,\n        model,\n        tools=effective_preset.tools,\n        retry_policy=retry_policy,\n        sleeper=sleeper,\n        run_guard=run_guard,\n        generation_settings=generation_settings,\n        history=(AgentMessage.text(Role.SYSTEM, prompt.text),) if no_save else (),\n    )\n    session = create_agent_session(\n        runtime,\n        store=effective_store,\n        session_id=session_id,\n        fork_from=fork_from,\n        no_save=no_save,\n        compaction_policy=compaction_policy,\n        compaction_strategy=compaction_strategy,\n        token_estimator=token_estimator,\n        extensions=extensions,\n        prompt_hashes=prompt.prompt_hashes,\n        resource_hashes=prompt.resource_hashes,\n        observer=effective_observer,\n        strict_tracing=strict_tracing,\n    )\n    if not no_save and session_id is None:\n        # The effective system context is settled once into each new durable Session.\n        assert effective_store is not None and session.session_id is not None\n        with effective_store.writer(session.session_id) as writer:\n            writer.append((AgentMessage.text(Role.SYSTEM, prompt.text),))\n        runtime.restore_history((AgentMessage.text(Role.SYSTEM, prompt.text),))\n    return session\n'

In [ ]:
INTERFACES_SOURCE = '"""Terminal-neutral result, event, and live Session adapter contracts."""\n\nfrom __future__ import annotations\n\nfrom collections.abc import AsyncIterator, Mapping\nfrom dataclasses import dataclass, fields, is_dataclass\nfrom enum import Enum\nfrom typing import Literal\n\nfrom .model import TextContent\nfrom .runtime import RuntimeEvent, TerminalStatus\nfrom .session import (\n    AgentSession,\n    InputKind,\n    PendingInput,\n    SessionRunHandle,\n    SessionRunResult,\n)\n\n\nRUN_RESULT_SCHEMA_VERSION = {"major": 1, "minor": 0}\nEVENT_ENVELOPE_SCHEMA_VERSION = {"major": 1, "minor": 0}\n\n\ndef _json_value(value: object) -> object:\n    if value is None or isinstance(value, (str, bool, int, float)):\n        return value\n    if isinstance(value, Enum):\n        return value.value\n    if is_dataclass(value):\n        return {\n            field.name: _json_value(getattr(value, field.name))\n            for field in fields(value)\n        }\n    if isinstance(value, Mapping):\n        return {str(key): _json_value(item) for key, item in value.items()}\n    if isinstance(value, (list, tuple)):\n        return [_json_value(item) for item in value]\n    return str(value)\n\n\n@dataclass(frozen=True, slots=True)\nclass RunResult:\n    run_id: str\n    session_id: str | None\n    status: str\n    stop_reason: str\n    final_text: str\n    usage: object | None\n    attempts: int\n    trace_complete: bool\n    trace_error: str | None = None\n    schema: Literal["agent_harness.run_result"] = "agent_harness.run_result"\n\n    @classmethod\n    def from_session(\n        cls,\n        handle: SessionRunHandle,\n        session_id: str | None,\n        result: SessionRunResult,\n    ) -> "RunResult":\n        outcome = result.outcome\n        final_text = "".join(\n            block.text for block in outcome.message.content if isinstance(block, TextContent)\n        )\n        return cls(\n            handle.run_id,\n            session_id,\n            outcome.status.value,\n            outcome.stop_reason.value,\n            final_text,\n            _json_value(outcome.usage),\n            outcome.attempts,\n            result.trace_complete,\n            result.trace_error,\n        )\n\n    def to_dict(self) -> dict[str, object]:\n        return {\n            "schema": self.schema,\n            "schema_version": dict(RUN_RESULT_SCHEMA_VERSION),\n            "run_id": self.run_id,\n            "session_id": self.session_id,\n            "status": self.status,\n            "stop_reason": self.stop_reason,\n            "final_text": self.final_text,\n            "usage": self.usage,\n            "attempts": self.attempts,\n            "trace_complete": self.trace_complete,\n            "trace_error": self.trace_error,\n        }\n\n\n@dataclass(frozen=True, slots=True)\nclass EventEnvelope:\n    sequence: int\n    type: str\n    session_id: str | None\n    run_id: str\n    payload: Mapping[str, object]\n    schema: Literal["agent_harness.event"] = "agent_harness.event"\n\n    @classmethod\n    def from_runtime(\n        cls, event: RuntimeEvent, *, session_id: str | None\n    ) -> "EventEnvelope":\n        if event.run_id is None:\n            raise ValueError("Runtime Event is missing its Run id")\n        return cls(\n            event.sequence,\n            event.type.value,\n            session_id,\n            event.run_id,\n            {\n                "attempt": event.attempt,\n                "operation": event.operation.value,\n                "model_event": _json_value(event.model_event),\n                "error": _json_value(event.error),\n                "retry_delay_seconds": event.retry_delay_seconds,\n                "partial_text": event.partial_text,\n                "partial_usage": _json_value(event.partial_usage),\n                "tool_call_id": event.tool_call_id,\n                "tool_name": event.tool_name,\n                "tool_arguments": _json_value(event.tool_arguments),\n                "tool_result": _json_value(event.tool_result),\n                "extension_name": event.extension_name,\n                "diagnostic": event.diagnostic,\n                "snapshot_fingerprint": event.snapshot_fingerprint,\n            },\n        )\n\n    @classmethod\n    def run_end(cls, result: RunResult, sequence: int) -> "EventEnvelope":\n        return cls(\n            sequence,\n            "run_end",\n            result.session_id,\n            result.run_id,\n            result.to_dict(),\n        )\n\n    def to_dict(self) -> dict[str, object]:\n        return {\n            "schema": self.schema,\n            "schema_version": dict(EVENT_ENVELOPE_SCHEMA_VERSION),\n            "sequence": self.sequence,\n            "type": self.type,\n            "session_id": self.session_id,\n            "run_id": self.run_id,\n            "payload": dict(self.payload),\n        }\n\n\nclass NonTerminalAdapter:\n    """A UI-neutral controller proving the live Session seam is reusable."""\n\n    def __init__(self, session: AgentSession) -> None:\n        if not isinstance(session, AgentSession):\n            raise TypeError("session must be an AgentSession")\n        self.session = session\n        self._handle: SessionRunHandle | None = None\n        self._restored_inputs: tuple[PendingInput, ...] = ()\n\n    @property\n    def snapshot(self):\n        if self._handle is None:\n            raise RuntimeError("no Run has started")\n        return self._handle.snapshot\n\n    @property\n    def restored_inputs(self) -> tuple[PendingInput, ...]:\n        return self._restored_inputs\n\n    def start(self, prompt: str) -> None:\n        if self._handle is not None:\n            raise RuntimeError("the adapter already controls a Run")\n        self._handle = self.session.start(prompt)\n        restored = self._restored_inputs\n        self._restored_inputs = ()\n        for pending in restored:\n            text = "".join(\n                block.text\n                for block in pending.message.content\n                if isinstance(block, TextContent)\n            )\n            if pending.kind is InputKind.STEERING:\n                self.session.steer(text)\n            else:\n                self.session.follow_up(text)\n\n    async def events(self) -> AsyncIterator[EventEnvelope]:\n        if self._handle is None:\n            raise RuntimeError("no Run has started")\n        async for event in self._handle.events():\n            yield EventEnvelope.from_runtime(event, session_id=self.session.session_id)\n\n    def steer(self, message: str) -> None:\n        self.session.steer(message)\n\n    def follow_up(self, message: str) -> None:\n        self.session.follow_up(message)\n\n    def cancel(self) -> None:\n        if self._handle is None:\n            raise RuntimeError("no Run has started")\n        self._handle.cancel()\n\n    async def result(self) -> RunResult:\n        if self._handle is None:\n            raise RuntimeError("no Run has started")\n        result = await self._handle.result()\n        converted = RunResult.from_session(\n            self._handle, self.session.session_id, result\n        )\n        self._restored_inputs = result.pending_inputs\n        self._handle = None\n        return converted\n\n    async def command(self, name: str, arguments: str = "") -> object:\n        return await self.session.execute_command(name, arguments)\n\n\ndef exit_code(status: str | TerminalStatus) -> int:\n    accepted = status.value if isinstance(status, TerminalStatus) else status\n    if accepted == TerminalStatus.COMPLETED.value:\n        return 0\n    if accepted == TerminalStatus.MODEL_ERROR.value:\n        return 3\n    if accepted == TerminalStatus.CANCELLED.value:\n        return 130\n    if accepted in {\n        TerminalStatus.MAX_TURNS.value,\n        TerminalStatus.MAX_TOOL_CALLS.value,\n        TerminalStatus.TIMEOUT.value,\n        TerminalStatus.MAX_TOTAL_TOKENS.value,\n    }:\n        return 5\n    return 4\n'

In [ ]:
CLI_SOURCE = '"""Omega command-line Interface Adapters over public AgentSession contracts."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport asyncio\nfrom collections.abc import Callable, Mapping, Sequence\nimport json\nimport os\nfrom pathlib import Path\nimport sys\nimport threading\nfrom typing import TextIO\n\nfrom .api import create_coding_session\nfrom .interfaces import EventEnvelope, NonTerminalAdapter, RunResult, exit_code\nfrom .model import ModelAdapter, ModelSpec, OpenAICompatibleAdapter, OpenAICompatibleConfig\nfrom .observability import LocalWorkspace\nfrom .runtime import RunGuard\n\n\nAdapterFactory = Callable[[OpenAICompatibleConfig], ModelAdapter]\n\n\nclass CLIConfigurationError(ValueError):\n    """Command-line and environment configuration is incomplete or inconsistent."""\n\n\ndef build_parser() -> argparse.ArgumentParser:\n    parser = argparse.ArgumentParser(prog="omega")\n    subcommands = parser.add_subparsers(dest="command", required=True)\n    execute = subcommands.add_parser("exec", help="run one prompt")\n    execute.add_argument("prompt")\n    _add_runtime_arguments(execute)\n    execute.add_argument("--format", choices=("text", "json", "jsonl"), default="text")\n    execute.add_argument("--quiet", action="store_true")\n\n    chat = subcommands.add_parser("chat", help="run a line-oriented conversation")\n    _add_runtime_arguments(chat)\n    chat.add_argument("--continue", dest="continue_latest", action="store_true")\n\n    sessions = subcommands.add_parser("sessions", help="inspect durable Sessions")\n    session_commands = sessions.add_subparsers(dest="sessions_command", required=True)\n    for name in ("list", "show", "export", "fork"):\n        command = session_commands.add_parser(name)\n        if name != "list":\n            command.add_argument("session_id")\n        if name == "fork":\n            command.add_argument("--entry")\n        _add_storage_arguments(command)\n\n    runs = subcommands.add_parser("runs", help="inspect durable Runs")\n    run_commands = runs.add_subparsers(dest="runs_command", required=True)\n    run_list = run_commands.add_parser("list")\n    run_list.add_argument("--session")\n    run_list.add_argument("--status")\n    _add_storage_arguments(run_list)\n    for name in ("show", "export"):\n        command = run_commands.add_parser(name)\n        command.add_argument("run_id")\n        _add_storage_arguments(command)\n    annotate = run_commands.add_parser("annotate")\n    annotate.add_argument("run_id")\n    annotate.add_argument("--namespace", required=True)\n    annotate.add_argument("--payload", required=True)\n    _add_storage_arguments(annotate)\n    prune = run_commands.add_parser("prune")\n    prune.add_argument("--session")\n    prune.add_argument("--status")\n    prune.add_argument("--apply", action="store_true")\n    _add_storage_arguments(prune)\n    return parser\n\n\ndef _add_runtime_arguments(parser: argparse.ArgumentParser) -> None:\n    parser.add_argument("--workspace", default=".")\n    parser.add_argument("--storage-root")\n    parser.add_argument("--session")\n    parser.add_argument("--no-save", action="store_true")\n    parser.add_argument("--model")\n    parser.add_argument("--base-url")\n    parser.add_argument("--context-window", type=int)\n    parser.add_argument("--max-output-tokens", type=int, default=4096)\n    parser.add_argument("--max-turns", type=int)\n    parser.add_argument("--max-tool-calls", type=int)\n    parser.add_argument("--timeout", type=float)\n    parser.add_argument("--max-total-tokens", type=int)\n\n\ndef _add_storage_arguments(parser: argparse.ArgumentParser) -> None:\n    parser.add_argument("--workspace", default=".")\n    parser.add_argument("--storage-root")\n\n\ndef _configuration(\n    arguments: argparse.Namespace,\n    environ: Mapping[str, str],\n) -> tuple[OpenAICompatibleConfig, ModelSpec, RunGuard | None]:\n    api_key = environ.get("OPENAI_API_KEY", "")\n    if not api_key:\n        raise CLIConfigurationError("OPENAI_API_KEY is required")\n    model_id = arguments.model or environ.get("OPENAI_MODEL", "")\n    if not model_id:\n        raise CLIConfigurationError("provide --model or OPENAI_MODEL")\n    base_url = (\n        arguments.base_url\n        or environ.get("OPENAI_BASE_URL")\n        or "https://api.openai.com/v1"\n    )\n    config = OpenAICompatibleConfig(base_url=base_url, api_key=api_key)\n    model = ModelSpec(\n        model_id,\n        context_window=arguments.context_window,\n        max_output_tokens=arguments.max_output_tokens,\n    )\n    guard_values = (\n        arguments.max_turns,\n        arguments.max_tool_calls,\n        arguments.timeout,\n        arguments.max_total_tokens,\n    )\n    guard = None\n    if any(value is not None for value in guard_values):\n        guard = RunGuard(\n            max_turns=arguments.max_turns,\n            max_tool_calls=arguments.max_tool_calls,\n            timeout_seconds=arguments.timeout,\n            max_total_tokens=arguments.max_total_tokens,\n        )\n    return config, model, guard\n\n\nasync def _exec(\n    arguments: argparse.Namespace,\n    *,\n    config: OpenAICompatibleConfig,\n    model: ModelSpec,\n    guard: RunGuard | None,\n    stdout: TextIO,\n    stderr: TextIO,\n    adapter_factory: AdapterFactory,\n) -> int:\n    workspace = Path(arguments.workspace).resolve()\n    if arguments.no_save and arguments.session is not None:\n        raise CLIConfigurationError("--no-save cannot continue a durable Session")\n    adapter = adapter_factory(config)\n    session = create_coding_session(\n        adapter,\n        model,\n        workspace=workspace,\n        storage_root=arguments.storage_root,\n        session_id=arguments.session,\n        no_save=arguments.no_save,\n        trace_secrets=(config.api_key, *config.headers.values()),\n        run_guard=guard,\n    )\n    handle = session.start(arguments.prompt)\n    envelopes: list[EventEnvelope] = []\n    async for event in handle.events():\n        envelope = EventEnvelope.from_runtime(event, session_id=session.session_id)\n        envelopes.append(envelope)\n        if (\n            arguments.format == "text"\n            and not arguments.quiet\n            and event.type.value in {"retry_scheduled", "tool_call_start", "run_cancelled"}\n        ):\n            print(event.type.value, file=stderr)\n    session_result = await handle.result()\n    result = RunResult.from_session(handle, session.session_id, session_result)\n    if arguments.format == "text":\n        stdout.write(result.final_text)\n        if result.final_text and not result.final_text.endswith("\\n"):\n            stdout.write("\\n")\n    elif arguments.format == "json":\n        stdout.write(json.dumps(result.to_dict(), ensure_ascii=False, sort_keys=True) + "\\n")\n    else:\n        for envelope in envelopes:\n            stdout.write(\n                json.dumps(envelope.to_dict(), ensure_ascii=False, sort_keys=True) + "\\n"\n            )\n        ending = EventEnvelope.run_end(\n            result,\n            max((item.sequence for item in envelopes), default=0) + 1,\n        )\n        stdout.write(json.dumps(ending.to_dict(), ensure_ascii=False, sort_keys=True) + "\\n")\n    return exit_code(result.status)\n\n\ndef _latest_session_id(\n    workspace: Path,\n    storage_root: str | Path | None,\n) -> str:\n    local = LocalWorkspace.open(workspace, storage_root=storage_root)\n    directory = local.sessions.sessions_directory\n    candidates = tuple(directory.glob("*.jsonl")) if directory.is_dir() else ()\n    if not candidates:\n        raise CLIConfigurationError("no durable Session exists for this workspace")\n    return max(candidates, key=lambda path: path.stat().st_mtime_ns).stem\n\n\nasync def _chat(\n    arguments: argparse.Namespace,\n    *,\n    config: OpenAICompatibleConfig,\n    model: ModelSpec,\n    guard: RunGuard | None,\n    stdout: TextIO,\n    stderr: TextIO,\n    input_stream: TextIO,\n    adapter_factory: AdapterFactory,\n) -> int:\n    workspace = Path(arguments.workspace).resolve()\n    if arguments.continue_latest and arguments.session is not None:\n        raise CLIConfigurationError("--continue and --session are mutually exclusive")\n    if arguments.no_save and (arguments.continue_latest or arguments.session is not None):\n        raise CLIConfigurationError("--no-save cannot continue a durable Session")\n    session_id = arguments.session\n    if arguments.continue_latest:\n        session_id = _latest_session_id(workspace, arguments.storage_root)\n    session = create_coding_session(\n        adapter_factory(config),\n        model,\n        workspace=workspace,\n        storage_root=arguments.storage_root,\n        session_id=session_id,\n        no_save=arguments.no_save,\n        trace_secrets=(config.api_key, *config.headers.values()),\n        run_guard=guard,\n    )\n    controller = NonTerminalAdapter(session)\n    lines: asyncio.Queue[str | None] = asyncio.Queue()\n    loop = asyncio.get_running_loop()\n\n    def read_lines() -> None:\n        while True:\n            line = input_stream.readline()\n            if line == "":\n                try:\n                    loop.call_soon_threadsafe(lines.put_nowait, None)\n                except RuntimeError:\n                    pass\n                return\n            try:\n                loop.call_soon_threadsafe(lines.put_nowait, line)\n            except RuntimeError:\n                return\n\n    threading.Thread(target=read_lines, daemon=True).start()\n\n    async def settle_active() -> RunResult:\n        async for event in controller.events():\n            if event.type in {\n                "retry_scheduled",\n                "tool_call_start",\n                "tool_call_end",\n                "run_cancelled",\n            }:\n                tool_name = event.payload.get("tool_name")\n                detail = f" {tool_name}" if tool_name else ""\n                print(f"[{event.type}]{detail}", file=stderr)\n        return await controller.result()\n\n    active: asyncio.Task[RunResult] | None = None\n    pending_line: str | None = None\n    wait_for_result = False\n    quit_after_run = False\n    while True:\n        if active is not None:\n            input_task: asyncio.Task[str | None] | None = None\n            waiters: set[asyncio.Task[object]] = {active}\n            if not wait_for_result:\n                input_task = asyncio.create_task(lines.get())\n                waiters.add(input_task)\n            completed, _ = await asyncio.wait(\n                waiters, return_when=asyncio.FIRST_COMPLETED\n            )\n            if active in completed:\n                result = active.result()\n                if result.final_text:\n                    print(f"assistant: {result.final_text}", file=stdout)\n                active = None\n                wait_for_result = False\n                if input_task is not None:\n                    if input_task.done():\n                        consumed = input_task.result()\n                        if consumed is None:\n                            quit_after_run = True\n                        else:\n                            pending_line = consumed\n                    else:\n                        input_task.cancel()\n                if quit_after_run:\n                    return 0\n                if result.status == "cancelled":\n                    continue\n                if result.status != "completed":\n                    return exit_code(result.status)\n                continue\n            assert input_task is not None\n            incoming = input_task.result()\n            if incoming is None:\n                wait_for_result = True\n                quit_after_run = True\n                continue\n            control = incoming.strip()\n            if not control:\n                continue\n            if control == "/wait":\n                wait_for_result = True\n            elif control == "/quit":\n                controller.cancel()\n                wait_for_result = True\n                quit_after_run = True\n            elif control == "/cancel":\n                controller.cancel()\n                wait_for_result = True\n            elif control.startswith("/steer "):\n                controller.steer(control.removeprefix("/steer "))\n            elif control.startswith("/follow-up "):\n                controller.follow_up(control.removeprefix("/follow-up "))\n            elif control.startswith("/"):\n                print("chat commands require an idle Session", file=stderr)\n            else:\n                controller.follow_up(control)\n            continue\n\n        line = pending_line\n        pending_line = None\n        if line is None:\n            line = await lines.get()\n        if line is None:\n            return 0\n        text = line.strip()\n        if not text:\n            continue\n        if text == "/quit":\n            return 0\n        if text == "/wait":\n            continue\n        if text == "/cancel" or text.startswith(("/steer ", "/follow-up ")):\n            print("no active Run", file=stderr)\n            continue\n        if text.startswith("/compact"):\n            focus = text.removeprefix("/compact").strip() or None\n            await session.compact(focus)\n            continue\n        if text.startswith("/"):\n            command, _, command_arguments = text[1:].partition(" ")\n            rendered = await controller.command(command, command_arguments)\n            if rendered is not None:\n                print(rendered, file=stdout)\n            continue\n        controller.start(text)\n        active = asyncio.create_task(settle_active())\n    return 0\n\n\ndef _session_summaries(local: LocalWorkspace) -> list[dict[str, object]]:\n    directory = local.sessions.sessions_directory\n    if not directory.is_dir():\n        return []\n    summaries: list[dict[str, object]] = []\n    for path in sorted(directory.glob("*.jsonl")):\n        state = local.sessions.read(path.stem)\n        summaries.append(\n            {\n                "session_id": state.session_id,\n                "entries": len(state.entries),\n                "active_leaf_id": state.active_leaf_id,\n                "recovery_warning": (\n                    None\n                    if state.recovery_warning is None\n                    else state.recovery_warning.code.value\n                ),\n                "modified_ns": path.stat().st_mtime_ns,\n            }\n        )\n    def modified(item: Mapping[str, object]) -> int:\n        value = item["modified_ns"]\n        assert isinstance(value, int)\n        return value\n\n    return sorted(summaries, key=modified, reverse=True)\n\n\ndef _storage_command(arguments: argparse.Namespace, stdout: TextIO) -> int:\n    local = LocalWorkspace.open(\n        Path(arguments.workspace).resolve(), storage_root=arguments.storage_root\n    )\n    if arguments.command == "sessions":\n        if arguments.sessions_command == "list":\n            payload: object = _session_summaries(local)\n        elif arguments.sessions_command == "show":\n            state = local.sessions.read(arguments.session_id)\n            payload = {\n                "session_id": state.session_id,\n                "entries": len(state.entries),\n                "active_leaf_id": state.active_leaf_id,\n                "messages": len(state.history()),\n                "compactions": len(state.compactions()),\n                "custom_entries": len(state.custom_entries()),\n            }\n        elif arguments.sessions_command == "export":\n            stdout.write(local.sessions.path_for(arguments.session_id).read_text(encoding="utf-8"))\n            return 0\n        else:\n            source = local.sessions.read(arguments.session_id)\n            leaf = arguments.entry or source.active_leaf_id\n            history = source.history(leaf)\n            forked = local.sessions.create()\n            if history:\n                with local.sessions.writer(forked.session_id) as writer:\n                    writer.append(history)\n            payload = {\n                "session_id": forked.session_id,\n                "forked_from": source.session_id,\n                "entry_id": leaf,\n            }\n    else:\n        if arguments.runs_command == "list":\n            payload = [\n                {\n                    "run_id": item.run_id,\n                    "session_id": item.session_id,\n                    "status": item.status,\n                    "started_at": item.started_at,\n                    "finished_at": item.finished_at,\n                    "evidence_complete": item.evidence_complete,\n                }\n                for item in local.runs.list(\n                    session_id=arguments.session, status=arguments.status\n                )\n            ]\n        elif arguments.runs_command == "show":\n            summary = local.runs.summary(arguments.run_id)\n            payload = {\n                "run_id": summary.run_id,\n                "session_id": summary.session_id,\n                "status": summary.status,\n                "started_at": summary.started_at,\n                "finished_at": summary.finished_at,\n                "evidence_complete": summary.evidence_complete,\n                "records": list(local.runs.records(arguments.run_id)),\n                "annotations": list(local.runs.annotations(arguments.run_id)),\n            }\n        elif arguments.runs_command == "export":\n            stdout.write(local.runs.export(arguments.run_id))\n            return 0\n        elif arguments.runs_command == "annotate":\n            try:\n                annotation = json.loads(arguments.payload)\n            except json.JSONDecodeError as error:\n                raise CLIConfigurationError("--payload must be valid JSON") from error\n            if not isinstance(annotation, dict):\n                raise CLIConfigurationError("--payload must be a JSON object")\n            local.runs.annotate(arguments.run_id, arguments.namespace, annotation)\n            payload = {"run_id": arguments.run_id, "annotated": True}\n        else:\n            preview = local.runs.prune_preview(\n                session_id=arguments.session, status=arguments.status\n            )\n            if arguments.apply:\n                local.runs.prune(preview)\n            payload = {\n                "run_ids": list(preview.run_ids),\n                "bytes": preview.bytes,\n                "applied": bool(arguments.apply),\n            }\n    stdout.write(json.dumps(payload, ensure_ascii=False, sort_keys=True) + "\\n")\n    return 0\n\n\ndef main(\n    argv: Sequence[str] | None = None,\n    *,\n    environ: Mapping[str, str] | None = None,\n    stdout: TextIO | None = None,\n    stderr: TextIO | None = None,\n    adapter_factory: AdapterFactory = OpenAICompatibleAdapter,\n    input_stream: TextIO | None = None,\n) -> int:\n    """Run Omega without hiding terminal or environment state in the library."""\n\n    accepted_environ = os.environ if environ is None else environ\n    output = sys.stdout if stdout is None else stdout\n    diagnostics = sys.stderr if stderr is None else stderr\n    terminal_input = sys.stdin if input_stream is None else input_stream\n    try:\n        arguments = build_parser().parse_args(list(argv) if argv is not None else None)\n        if arguments.command in {"sessions", "runs"}:\n            return _storage_command(arguments, output)\n        config, model, guard = _configuration(arguments, accepted_environ)\n        if arguments.command == "exec":\n            operation = _exec(\n                    arguments,\n                    config=config,\n                    model=model,\n                    guard=guard,\n                    stdout=output,\n                    stderr=diagnostics,\n                    adapter_factory=adapter_factory,\n                )\n        elif arguments.command == "chat":\n            operation = _chat(\n                    arguments,\n                    config=config,\n                    model=model,\n                    guard=guard,\n                    stdout=output,\n                    stderr=diagnostics,\n                    input_stream=terminal_input,\n                    adapter_factory=adapter_factory,\n                )\n        else:\n            raise CLIConfigurationError(\n                f"{arguments.command} command is not implemented by this checkpoint"\n            )\n        return asyncio.run(operation)\n    except CLIConfigurationError as error:\n        print(f"omega: configuration error: {error}", file=diagnostics)\n        return 2\n    except KeyboardInterrupt:\n        print("omega: cancelled", file=diagnostics)\n        return 130\n    except Exception as error:\n        print(f"omega: infrastructure error: {type(error).__name__}: {error}", file=diagnostics)\n        return 4\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n'

In [ ]:
INIT_SOURCE = 'from .compaction import (\n    CharacterTokenEstimator,\n    CompactionCheckpoint,\n    CompactionPlan,\n    CompactionPolicy,\n    CompactionStrategy,\n    CompactionTrigger,\n    CompactionWarning,\n    CompactionWarningCode,\n    ResolvedCompactionPolicy,\n    StructuredSummary,\n    TokenEstimator,\n)\nfrom .model import (\n    AgentMessage,\n    ContentBlock,\n    ModelAdapter,\n    ModelAdapterError,\n    ModelEnd,\n    ModelError,\n    ModelErrorCode,\n    ModelEvent,\n    ModelMessage,\n    ModelOperation,\n    ModelProtocolError,\n    ModelRequest,\n    ModelResult,\n    ModelSpec,\n    ModelToolResultMessage,\n    OpenAICompatibleAdapter,\n    OpenAICompatibleConfig,\n    Role,\n    ScriptedModelAdapter,\n    StopReason,\n    TextContent,\n    TextDelta,\n    ToolCallContent,\n    ToolCallDelta,\n    UnsupportedContentError,\n    Usage,\n    UsageUpdate,\n    complete,\n    to_model_messages,\n)\nfrom .extensions import (\n    Extension,\n    ExtensionAPI,\n    ExtensionHost,\n    ExtensionInitializationError,\n    SubscriberRegistration,\n    HookContext,\n    HookExecutionError,\n    HookPoint,\n    HookRegistration,\n    CompactionHookRequest,\n    CommandExecutionError,\n    CommandRegistration,\n    CustomEntry,\n    RunAnnotation,\n    LifecycleWarning,\n    ReloadResult,\n    ReplacementRecord,\n)\nfrom .persistence import (\n    ConversationMessage,\n    JSONLSessionStore,\n    JSONLSessionWriter,\n    MemorySessionStore,\n    MemorySessionWriter,\n    MigrationResult,\n    RecoveryCode,\n    RecoveryWarning,\n    SESSION_SCHEMA_VERSION,\n    SchemaVersion,\n    SessionBusyError,\n    SessionEntry,\n    SessionStore,\n    SessionState,\n    SessionWriter,\n    UnsupportedSchemaVersionError,\n    migrate_session_file,\n)\nfrom .runtime import (\n    AgentRunHandle,\n    AgentRuntime,\n    AssistantOutcome,\n    EventType,\n    RetryPolicy,\n    RunSnapshot,\n    RunGuard,\n    RuntimeEvent,\n    Sleeper,\n    ContextOverflowRecovery,\n    SettledTurnHandler,\n    SummaryGeneration,\n    TerminalStatus,\n    TurnInput,\n)\nfrom .session import (\n    AgentSession,\n    CompactionResult,\n    InputKind,\n    PendingInput,\n    SessionRunHandle,\n    SessionRunResult,\n    RunObserver,\n    TracePersistenceError,\n    create_agent_session,\n)\nfrom .tools import (\n    AsyncioProcessOperations,\n    CompleteOutputKind,\n    CompleteOutputReference,\n    LocalToolExecutor,\n    PreparedToolCall,\n    ProcessResult,\n    Tool,\n    ToolErrorCode,\n    ToolExecutor,\n    ToolOutputBudget,\n    ToolResult,\n    ToolResultMessage,\n    TruncationDirection,\n    TruncationNotice,\n)\nfrom .resources import (\n    LoadedResources,\n    JSONProjectTrustStore,\n    MemoryProjectTrust,\n    PromptAssembler,\n    PromptAssembly,\n    ProjectTrust,\n    Resource,\n    ResourceEvidence,\n    ResourceKind,\n    ResourceLoader,\n    ResourceScope,\n)\nfrom .coding import (\n    ArtifactStore,\n    BashOperations,\n    CodingAgent,\n    CodingToolPreset,\n    FileArtifactStore,\n    WorkspaceBoundary,\n    WorkspaceBoundaryError,\n    create_coding_agent,\n    create_coding_tool_preset,\n    filter_sensitive_environment,\n    resolve_bash,\n)\nfrom .observability import (\n    ARTIFACT_SCHEMA_VERSION,\n    RUN_ANNOTATION_SCHEMA_VERSION,\n    RUN_TRACE_SCHEMA_VERSION,\n    JSONLRunStore,\n    LocalWorkspace,\n    PrunePreview,\n    RunArtifactStore,\n    RunStore,\n    RunSummary,\n    StandardTraceRedactor,\n    StandardTraceSink,\n    TraceRedactor,\n    UnsupportedRunSchemaVersionError,\n    WorkspaceIdentity,\n    migrate_run_trace,\n)\nfrom .api import create_coding_session, create_session\nfrom .interfaces import (\n    EVENT_ENVELOPE_SCHEMA_VERSION,\n    RUN_RESULT_SCHEMA_VERSION,\n    EventEnvelope,\n    NonTerminalAdapter,\n    RunResult,\n    exit_code,\n)\n\n__all__ = [name for name in globals() if not name.startswith("_")]\n'

In [ ]:
import importlib
import shutil
from tempfile import TemporaryDirectory

temporary_package = TemporaryDirectory(prefix='chapter-09-minimal-')
package = Path(temporary_package.name) / 'agent_harness'
shutil.copytree(CHAPTER_8 / 'src' / 'agent_harness', package)
for name, text in {
    'runtime.py': RUNTIME_SOURCE, 'session.py': SESSION_SOURCE,
    'observability.py': OBSERVABILITY_SOURCE, 'api.py': API_SOURCE,
    'interfaces.py': INTERFACES_SOURCE, 'cli.py': CLI_SOURCE,
    '__init__.py': INIT_SOURCE,
}.items():
    (package / name).write_text(text, encoding='utf-8')
sys.path.insert(0, temporary_package.name)
chapter9 = importlib.import_module('agent_harness')

temporary_workspace = TemporaryDirectory(prefix='chapter-09-workspace-')
workspace = Path(temporary_workspace.name)
adapter = chapter9.ScriptedModelAdapter([
    chapter9.TextDelta('offline version one'),
    chapter9.ModelEnd(chapter9.StopReason.COMPLETE),
])
session = chapter9.create_session(
    adapter, chapter9.ModelSpec('scripted/ch09-minimal'),
    workspace=workspace, no_save=True,
)
minimal = await session.run('finish the interface')
assert minimal.outcome.message.content[0].text == 'offline version one'
assert session.session_id is None

## Staged Construction

Stage one freezes factories and explicit configuration. Stage two groups transparent Session and Run files by Workspace Identity. Stage three adds RunResult/Event Envelope contracts and both terminal and non-terminal interface adapters.

In [ ]:
'[build-system]\nrequires = ["setuptools>=68"]\nbuild-backend = "setuptools.build_meta"\n\n[project]\nname = "agent-harness"\nversion = "1.0.0"\ndescription = "Reusable agent harness with Omega CLI and local Run evidence"\nreadme = "README.md"\nrequires-python = ">=3.11"\ndependencies = [\n  "jsonschema>=4.23,<5",\n  "openai>=1.40,<3",\n  "platformdirs>=4.2,<5",\n]\n\n[project.scripts]\nomega = "agent_harness.cli:main"\n\n[tool.setuptools.packages.find]\nwhere = ["src"]\n\n[tool.setuptools.package-data]\nagent_harness = ["py.typed"]\n\n[tool.pytest.ini_options]\ntestpaths = ["tests"]\n'

In [ ]:
'# Agent Harness — Chapter 9 / Version 1 Checkpoint\n\nThis final cumulative Checkpoint freezes the reusable Python interface and ships\nthe `omega` adapters over the same `AgentSession` contract. `create_session()`\nbuilds a general Session; `create_coding_session()` explicitly installs local\nresources plus `read`, `write`, `edit`, and a real host `bash`. Callers may use\ncompletion-oriented `await session.run(...)` or controllable\n`session.start(...)` with Events, snapshots, Steering, Follow-up, commands, and\ncoordinated cancellation.\n\nThe Bash adapter remains a real host process. Neither Bash nor Extensions are a sandbox guarantee;\nthe Workspace Boundary confines file adapters but is not a built-in process sandbox.\n\nThe library accepts a `ModelAdapter` or explicit `OpenAICompatibleConfig` plus\n`ModelSpec`; it never reads environment or dotenv state. The CLI alone translates\n`OPENAI_API_KEY`, `OPENAI_BASE_URL`, and `OPENAI_MODEL`. `--model` and\n`--base-url` override their non-secret environment equivalents. There is no\nAPI-key flag.\n\n```bash\nomega exec "inspect this workspace"\nomega exec "machine task" --format json\nomega exec "event stream" --format jsonl\nomega chat\nomega sessions list\nomega runs list\n```\n\n`omega exec` and `omega chat` create new durable Sessions by default. Continue\nonly with `--session ID` (or `omega chat --continue`). `--no-save` is the explicit\nprivacy mode and disables Session, Trace, Annotation, and Artifact persistence as\none boundary. Local records are grouped by canonical Workspace Identity: one\nJSONL file per Session and one directory per Run containing append-only Trace and\nAnnotation JSONL plus content-addressed Artifacts. Nothing is uploaded implicitly.\n\nText exec mode reserves stdout for final assistant text. JSON emits one versioned\n`RunResult`; JSONL emits ordered versioned Event Envelopes and always ends with\n`run_end`. Standard Trace records snapshots, platform and available Git evidence,\nattempts, retries, latency, usage provenance, Tool activity, errors, cancellation,\nand Compaction while redacting credential-shaped fields and bounding previews.\nOrdinary Trace failure marks evidence incomplete without changing semantic Run\nsuccess; strict tracing is opt-in.\n\nVersion one targets Python 3.11+ on Windows, Linux, and macOS. Runtime dependencies\nare only `openai`, `jsonschema`, and `platformdirs`, all within the accepted small\ncross-platform set. Jupyter, tests, documentation tooling, TUI/RPC/MCP stacks, and\nprovider catalogs are not runtime dependencies. The deliberate version-one\nexclusions are a full TUI, RPC/server mode, MCP, subagents, multimodal content,\noptimizer, remote Extension installation, model catalog, and built-in sandbox.\nThe real OpenAI-compatible smoke test remains explicit, credential-gated, and\nsupplementary to deterministic offline tests.\n'

In [ ]:
TEST_PUBLIC_API_SOURCE = 'from __future__ import annotations\n\nimport asyncio\nfrom pathlib import Path\n\nimport pytest\n\nfrom agent_harness import (\n    AgentSession,\n    LocalWorkspace,\n    MemorySessionStore,\n    ModelEnd,\n    ModelSpec,\n    OpenAICompatibleConfig,\n    ScriptedModelAdapter,\n    StopReason,\n    TextDelta,\n    ToolCallDelta,\n    TracePersistenceError,\n    create_session,\n    create_coding_session,\n)\n\n\ndef test_general_factory_persists_session_and_sanitized_run_evidence(\n    tmp_path: Path,\n) -> None:\n    workspace = tmp_path / "workspace"\n    storage_root = tmp_path / "data"\n    workspace.mkdir()\n    local = LocalWorkspace.open(workspace, storage_root=storage_root)\n    adapter = ScriptedModelAdapter(\n        [TextDelta("completed"), ModelEnd(StopReason.COMPLETE)]\n    )\n\n    session = create_session(\n        adapter,\n        ModelSpec("scripted/public"),\n        workspace=workspace,\n        storage_root=storage_root,\n        trace_secrets=("super-secret-value",),\n    )\n    result = asyncio.run(session.run("do not retain super-secret-value"))\n\n    assert isinstance(session, AgentSession)\n    assert session.session_id is not None\n    assert local.sessions.read(session.session_id).history()\n    summaries = local.runs.list(session_id=session.session_id)\n    assert len(summaries) == 1\n    assert summaries[0].status == "completed"\n    assert summaries[0].evidence_complete is True\n    assert result.trace_complete is True\n    assert "super-secret-value" not in local.runs.export(summaries[0].run_id)\n\n\ndef test_no_save_disables_every_persistence_family(tmp_path: Path) -> None:\n    workspace = tmp_path / "workspace"\n    storage_root = tmp_path / "must-not-exist"\n    workspace.mkdir()\n    session = create_session(\n        ScriptedModelAdapter([TextDelta("private"), ModelEnd(StopReason.COMPLETE)]),\n        ModelSpec("scripted/private"),\n        workspace=workspace,\n        storage_root=storage_root,\n        no_save=True,\n    )\n\n    result = asyncio.run(session.run("ephemeral prompt"))\n\n    assert session.session_id is None\n    assert result.outcome.message.content[0].text == "private"\n    assert not storage_root.exists()\n\n\ndef test_trace_failure_marks_evidence_incomplete_without_changing_success(\n    tmp_path: Path,\n) -> None:\n    class BrokenObserver:\n        def start_run(self, session_id, snapshot) -> None:\n            return None\n\n        def record_event(self, event) -> None:\n            raise OSError("disk unavailable")\n\n        def finish_run(self, session_id, run_id, result) -> None:\n            raise AssertionError("observer must stop after its first failure")\n\n        def mark_incomplete(self, run_id, reason) -> None:\n            return None\n\n    workspace = tmp_path / "workspace"\n    workspace.mkdir()\n    session = create_session(\n        ScriptedModelAdapter([TextDelta("still good"), ModelEnd(StopReason.COMPLETE)]),\n        ModelSpec("scripted/trace-failure"),\n        workspace=workspace,\n        session_store=MemorySessionStore(),\n        observer=BrokenObserver(),\n    )\n\n    result = asyncio.run(session.run("continue semantically"))\n\n    assert result.outcome.status.value == "completed"\n    assert result.trace_complete is False\n    assert result.trace_error == "OSError"\n\n\ndef test_strict_tracing_turns_evidence_failure_into_infrastructure_failure(\n    tmp_path: Path,\n) -> None:\n    class BrokenObserver:\n        def start_run(self, session_id, snapshot) -> None:\n            return None\n\n        def record_event(self, event) -> None:\n            raise OSError("disk unavailable")\n\n        def finish_run(self, session_id, run_id, result) -> None:\n            return None\n\n        def mark_incomplete(self, run_id, reason) -> None:\n            return None\n\n    workspace = tmp_path / "workspace"\n    workspace.mkdir()\n    session = create_session(\n        ScriptedModelAdapter([TextDelta("semantic output"), ModelEnd(StopReason.COMPLETE)]),\n        ModelSpec("scripted/strict-trace"),\n        workspace=workspace,\n        session_store=MemorySessionStore(),\n        observer=BrokenObserver(),\n        strict_tracing=True,\n    )\n\n    with pytest.raises(TracePersistenceError, match="record_event"):\n        asyncio.run(session.run("strict evidence"))\n\n\ndef test_coding_factory_installs_resources_and_tools_over_same_session_interface(\n    tmp_path: Path,\n) -> None:\n    async def scenario() -> None:\n        workspace = tmp_path / "workspace"\n        workspace.mkdir()\n        session = create_coding_session(\n            ScriptedModelAdapter(\n                [\n                    [\n                        ToolCallDelta(\n                            0,\n                            "write-1",\n                            "write",\n                            \'{"path":"answer.txt","content":"42"}\',\n                        ),\n                        ModelEnd(StopReason.TOOL_USE),\n                    ],\n                    [TextDelta("written"), ModelEnd(StopReason.COMPLETE)],\n                ]\n            ),\n            ModelSpec("scripted/coding"),\n            workspace=workspace,\n            no_save=True,\n        )\n\n        handle = session.start("write the answer")\n        result = await handle.result()\n\n        assert isinstance(session, AgentSession)\n        assert [tool.name for tool in handle.snapshot.tools] == [\n            "read",\n            "write",\n            "edit",\n            "bash",\n        ]\n        assert result.outcome.message.content[0].text == "written"\n        assert (workspace / "answer.txt").read_text() == "42"\n\n    asyncio.run(scenario())\n\n\ndef test_library_transport_configuration_is_explicit_and_ignores_environment(\n    tmp_path: Path, monkeypatch: pytest.MonkeyPatch\n) -> None:\n    workspace = tmp_path / "workspace"\n    workspace.mkdir()\n    monkeypatch.setenv("OPENAI_API_KEY", "ambient-key-must-not-be-read")\n    monkeypatch.setenv("OPENAI_BASE_URL", "https://ambient.invalid/v1")\n    monkeypatch.setenv("OPENAI_MODEL", "ambient-model")\n\n    session = create_session(\n        OpenAICompatibleConfig(\n            base_url="https://explicit.example/v1",\n            api_key="explicit-key",\n        ),\n        ModelSpec("explicit-model"),\n        workspace=workspace,\n        no_save=True,\n    )\n\n    assert isinstance(session, AgentSession)\n    assert session.session_id is None\n'

In [ ]:
TEST_CLI_SOURCE = 'from __future__ import annotations\n\nimport asyncio\nimport io\nimport json\nfrom pathlib import Path\n\nfrom agent_harness import (\n    LocalWorkspace,\n    ModelEnd,\n    ModelSpec,\n    ScriptedModelAdapter,\n    StopReason,\n    TextDelta,\n    create_session,\n)\nfrom agent_harness.cli import build_parser, main\n\n\ndef test_exec_json_uses_flag_precedence_and_persists_by_default(\n    tmp_path: Path,\n) -> None:\n    workspace = tmp_path / "workspace"\n    storage = tmp_path / "data"\n    workspace.mkdir()\n    stdout = io.StringIO()\n    stderr = io.StringIO()\n    captured = []\n\n    def adapter_factory(config):\n        captured.append(config)\n        return ScriptedModelAdapter(\n            [TextDelta("machine result"), ModelEnd(StopReason.COMPLETE)]\n        )\n\n    code = main(\n        [\n            "exec",\n            "finish it",\n            "--format",\n            "json",\n            "--workspace",\n            str(workspace),\n            "--storage-root",\n            str(storage),\n            "--model",\n            "flag-model",\n            "--base-url",\n            "https://flag.example/v1",\n        ],\n        environ={\n            "OPENAI_API_KEY": "cli-secret",\n            "OPENAI_MODEL": "env-model",\n            "OPENAI_BASE_URL": "https://env.example/v1",\n        },\n        stdout=stdout,\n        stderr=stderr,\n        adapter_factory=adapter_factory,\n    )\n\n    payload = json.loads(stdout.getvalue())\n    assert code == 0\n    assert stderr.getvalue() == ""\n    assert payload["schema"] == "agent_harness.run_result"\n    assert payload["schema_version"] == {"major": 1, "minor": 0}\n    assert payload["status"] == "completed"\n    assert payload["final_text"] == "machine result"\n    assert payload["session_id"]\n    assert payload["run_id"]\n    assert captured[0].base_url == "https://flag.example/v1"\n    assert captured[0].api_key == "cli-secret"\n    assert list(storage.rglob("trace.jsonl"))\n    assert "cli-secret" not in list(storage.rglob("trace.jsonl"))[0].read_text()\n\n\ndef test_exec_jsonl_emits_ordered_events_and_mandatory_run_end(\n    tmp_path: Path,\n) -> None:\n    workspace = tmp_path / "workspace"\n    workspace.mkdir()\n    stdout = io.StringIO()\n\n    code = main(\n        [\n            "exec",\n            "stream",\n            "--format",\n            "jsonl",\n            "--workspace",\n            str(workspace),\n            "--no-save",\n        ],\n        environ={"OPENAI_API_KEY": "x", "OPENAI_MODEL": "scripted/jsonl"},\n        stdout=stdout,\n        stderr=io.StringIO(),\n        adapter_factory=lambda config: ScriptedModelAdapter(\n            [TextDelta("one"), TextDelta(" two"), ModelEnd(StopReason.COMPLETE)]\n        ),\n    )\n\n    lines = [json.loads(line) for line in stdout.getvalue().splitlines()]\n    assert code == 0\n    assert lines[-1]["type"] == "run_end"\n    assert lines[-1]["payload"]["final_text"] == "one two"\n    assert [line["sequence"] for line in lines] == sorted(\n        line["sequence"] for line in lines\n    )\n    assert all(line["schema"] == "agent_harness.event" for line in lines)\n\n\ndef test_chat_runs_a_scripted_multi_turn_terminal_session(tmp_path: Path) -> None:\n    workspace = tmp_path / "workspace"\n    workspace.mkdir()\n    stdout = io.StringIO()\n    stderr = io.StringIO()\n    adapter = ScriptedModelAdapter(\n        [\n            [TextDelta("first answer"), ModelEnd(StopReason.COMPLETE)],\n            [TextDelta("second answer"), ModelEnd(StopReason.COMPLETE)],\n        ]\n    )\n\n    code = main(\n        ["chat", "--workspace", str(workspace), "--no-save"],\n        environ={"OPENAI_API_KEY": "x", "OPENAI_MODEL": "scripted/chat"},\n        stdout=stdout,\n        stderr=stderr,\n        input_stream=io.StringIO(\n            "first question\\n/wait\\nsecond question\\n/wait\\n/quit\\n"\n        ),\n        adapter_factory=lambda config: adapter,\n    )\n\n    assert code == 0\n    assert stdout.getvalue().splitlines() == [\n        "assistant: first answer",\n        "assistant: second answer",\n    ]\n    assert stderr.getvalue() == ""\n    assert len(adapter.received_requests) == 2\n\n\ndef test_session_and_run_commands_manage_local_evidence_without_credentials(\n    tmp_path: Path,\n) -> None:\n    workspace = tmp_path / "workspace"\n    storage = tmp_path / "data"\n    workspace.mkdir()\n    session = create_session(\n        ScriptedModelAdapter([TextDelta("kept"), ModelEnd(StopReason.COMPLETE)]),\n        ModelSpec("scripted/catalog"),\n        workspace=workspace,\n        storage_root=storage,\n    )\n    asyncio_result = asyncio.run(session.run("persist"))\n    assert asyncio_result.outcome.message.content[0].text == "kept"\n    local = LocalWorkspace.open(workspace, storage_root=storage)\n    run_id = local.runs.list()[0].run_id\n\n    def invoke(arguments: list[str]) -> tuple[int, str, str]:\n        out, err = io.StringIO(), io.StringIO()\n        code = main(arguments, environ={}, stdout=out, stderr=err)\n        return code, out.getvalue(), err.getvalue()\n\n    common = ["--workspace", str(workspace), "--storage-root", str(storage)]\n    code, output, error = invoke(["sessions", "list", *common])\n    assert code == 0 and error == ""\n    assert json.loads(output)[0]["session_id"] == session.session_id\n\n    code, output, error = invoke(\n        ["runs", "annotate", run_id, "--namespace", "review.outcome", "--payload", \'{"score":1}\', *common]\n    )\n    assert code == 0 and error == ""\n    assert local.runs.annotations(run_id)[0]["payload"] == {"score": 1}\n\n    code, output, error = invoke(["runs", "prune", "--status", "completed", *common])\n    preview = json.loads(output)\n    assert code == 0 and error == ""\n    assert preview["applied"] is False\n    assert preview["run_ids"] == [run_id]\n    assert local.runs.summary(run_id).status == "completed"\n\n    code, output, error = invoke(["sessions", "fork", session.session_id, *common])\n    forked = json.loads(output)["session_id"]\n    assert code == 0 and error == ""\n    assert forked != session.session_id\n    assert local.sessions.read(forked).history() == local.sessions.read(session.session_id).history()\n\n\ndef test_cli_has_no_api_key_flag_and_distinguishes_configuration_and_model_errors(\n    tmp_path: Path,\n) -> None:\n    workspace = tmp_path / "workspace"\n    workspace.mkdir()\n    help_text = build_parser().format_help()\n    assert "--api-key" not in help_text\n\n    missing_error = io.StringIO()\n    assert main(\n        ["exec", "x", "--workspace", str(workspace), "--no-save"],\n        environ={},\n        stdout=io.StringIO(),\n        stderr=missing_error,\n    ) == 2\n    assert "OPENAI_API_KEY" in missing_error.getvalue()\n\n    incompatible_error = io.StringIO()\n    assert main(\n        [\n            "exec",\n            "x",\n            "--workspace",\n            str(workspace),\n            "--no-save",\n            "--session",\n            "existing",\n        ],\n        environ={"OPENAI_API_KEY": "x", "OPENAI_MODEL": "scripted/error"},\n        stdout=io.StringIO(),\n        stderr=incompatible_error,\n        adapter_factory=lambda config: ScriptedModelAdapter([]),\n    ) == 2\n    assert "cannot continue" in incompatible_error.getvalue()\n\n    output = io.StringIO()\n    assert main(\n        ["exec", "x", "--format", "json", "--workspace", str(workspace), "--no-save"],\n        environ={"OPENAI_API_KEY": "x", "OPENAI_MODEL": "scripted/error"},\n        stdout=output,\n        stderr=io.StringIO(),\n        adapter_factory=lambda config: ScriptedModelAdapter([TextDelta("partial")]),\n    ) == 3\n    assert json.loads(output.getvalue())["status"] == "model_error"\n\n\ndef test_exec_continues_only_the_explicit_session(tmp_path: Path) -> None:\n    workspace = tmp_path / "workspace"\n    storage = tmp_path / "data"\n    workspace.mkdir()\n    environment = {"OPENAI_API_KEY": "x", "OPENAI_MODEL": "scripted/continue"}\n\n    first_output = io.StringIO()\n    assert main(\n        ["exec", "first", "--format", "json", "--workspace", str(workspace), "--storage-root", str(storage)],\n        environ=environment,\n        stdout=first_output,\n        stderr=io.StringIO(),\n        adapter_factory=lambda config: ScriptedModelAdapter(\n            [TextDelta("first answer"), ModelEnd(StopReason.COMPLETE)]\n        ),\n    ) == 0\n    session_id = json.loads(first_output.getvalue())["session_id"]\n\n    second_output = io.StringIO()\n    assert main(\n        [\n            "exec",\n            "second",\n            "--format",\n            "json",\n            "--workspace",\n            str(workspace),\n            "--storage-root",\n            str(storage),\n            "--session",\n            session_id,\n        ],\n        environ=environment,\n        stdout=second_output,\n        stderr=io.StringIO(),\n        adapter_factory=lambda config: ScriptedModelAdapter(\n            [TextDelta("second answer"), ModelEnd(StopReason.COMPLETE)]\n        ),\n    ) == 0\n\n    assert json.loads(second_output.getvalue())["session_id"] == session_id\n    local = LocalWorkspace.open(workspace, storage_root=storage)\n    assert len(local.runs.list(session_id=session_id)) == 2\n    assert len(local.sessions.read(session_id).history()) == 5\n\n\ndef test_chat_applies_live_steering_cancellation_and_restored_follow_up(\n    tmp_path: Path,\n) -> None:\n    class ControllableAdapter:\n        def __init__(self) -> None:\n            self.requests = []\n\n        async def stream(self, request):\n            self.requests.append(request)\n            turn = len(self.requests)\n            if turn == 1:\n                await asyncio.Event().wait()\n            responses = {2: "retry", 3: "steered", 4: "restored"}\n            yield TextDelta(responses[turn])\n            yield ModelEnd(StopReason.COMPLETE)\n\n    workspace = tmp_path / "workspace"\n    workspace.mkdir()\n    adapter = ControllableAdapter()\n    stdout, stderr = io.StringIO(), io.StringIO()\n\n    code = main(\n        ["chat", "--workspace", str(workspace), "--no-save"],\n        environ={"OPENAI_API_KEY": "x", "OPENAI_MODEL": "scripted/live-chat"},\n        stdout=stdout,\n        stderr=stderr,\n        input_stream=io.StringIO(\n            "begin\\n/steer adjust\\n/follow-up later\\n/cancel\\n/wait\\n"\n            "retry prompt\\n/wait\\n/quit\\n"\n        ),\n        adapter_factory=lambda config: adapter,\n    )\n\n    assert code == 0\n    assert stdout.getvalue().splitlines() == ["assistant: restored"]\n    assert "[run_cancelled]" in stderr.getvalue()\n    assert len(adapter.requests) == 4\n    requests = repr(adapter.requests)\n    assert "adjust" in requests\n    assert "later" in requests\n'

In [ ]:
TEST_INTERFACES_SOURCE = 'from __future__ import annotations\n\nimport asyncio\nfrom pathlib import Path\n\nfrom agent_harness import (\n    ModelEnd,\n    ModelSpec,\n    NonTerminalAdapter,\n    Extension,\n    ScriptedModelAdapter,\n    StopReason,\n    TextDelta,\n    create_session,\n)\n\n\ndef test_non_terminal_adapter_drives_snapshot_events_steering_and_follow_up(\n    tmp_path: Path,\n) -> None:\n    async def scenario() -> None:\n        workspace = tmp_path / "workspace"\n        workspace.mkdir()\n        model = ScriptedModelAdapter(\n            [\n                [TextDelta("initial"), ModelEnd(StopReason.COMPLETE)],\n                [TextDelta("steered"), ModelEnd(StopReason.COMPLETE)],\n                [TextDelta("followed"), ModelEnd(StopReason.COMPLETE)],\n            ]\n        )\n        ui = NonTerminalAdapter(\n            create_session(\n                model,\n                ModelSpec("scripted/non-terminal"),\n                workspace=workspace,\n                no_save=True,\n            )\n        )\n\n        ui.start("begin")\n        snapshot = ui.snapshot\n        ui.steer("adjust now")\n        ui.follow_up("then continue")\n        events = [event async for event in ui.events()]\n        result = await ui.result()\n\n        assert snapshot.model.model_id == "scripted/non-terminal"\n        assert events[0].type == "agent_start"\n        assert [event.sequence for event in events] == list(range(1, len(events) + 1))\n        assert result.final_text == "followed"\n        assert len(model.received_requests) == 3\n        request_text = repr(model.received_requests)\n        assert "adjust now" in request_text\n        assert "then continue" in request_text\n\n    asyncio.run(scenario())\n\n\ndef test_non_terminal_adapter_executes_the_same_session_commands(tmp_path: Path) -> None:\n    async def scenario() -> None:\n        workspace = tmp_path / "workspace"\n        workspace.mkdir()\n\n        async def inspect(arguments: str) -> object:\n            return {"arguments": arguments}\n\n        def configure(api) -> None:\n            api.register_command("inspect", inspect)\n\n        ui = NonTerminalAdapter(\n            create_session(\n                ScriptedModelAdapter(\n                    [TextDelta("unused"), ModelEnd(StopReason.COMPLETE)]\n                ),\n                ModelSpec("scripted/commands"),\n                workspace=workspace,\n                no_save=True,\n                extensions=(Extension("terminal", "1", configure),),\n            )\n        )\n\n        assert await ui.command("inspect", "state") == {"arguments": "state"}\n\n    asyncio.run(scenario())\n\n\ndef test_non_terminal_adapter_coordinates_cancellation(tmp_path: Path) -> None:\n    async def scenario() -> None:\n        workspace = tmp_path / "workspace"\n        workspace.mkdir()\n        ui = NonTerminalAdapter(\n            create_session(\n                ScriptedModelAdapter(\n                    [\n                        [TextDelta("retry answer"), ModelEnd(StopReason.COMPLETE)],\n                        [TextDelta("restored answer"), ModelEnd(StopReason.COMPLETE)],\n                    ]\n                ),\n                ModelSpec("scripted/cancel"),\n                workspace=workspace,\n                no_save=True,\n            )\n        )\n\n        ui.start("cancel this")\n        ui.follow_up("restore this input")\n        ui.cancel()\n        result = await ui.result()\n\n        assert result.status == "cancelled"\n        assert result.stop_reason == "aborted"\n        assert ui.restored_inputs[0].message.content[0].text == "restore this input"\n\n        ui.start("retry")\n        restored = await ui.result()\n\n        assert restored.final_text == "restored answer"\n        assert ui.restored_inputs == ()\n\n    asyncio.run(scenario())\n'

In [ ]:
TEST_OBSERVABILITY_SOURCE = 'from __future__ import annotations\n\nimport asyncio\nfrom pathlib import Path\n\nimport pytest\n\nfrom agent_harness import (\n    LocalWorkspace,\n    ModelAdapterError,\n    ModelEnd,\n    ModelError,\n    ModelErrorCode,\n    ModelSpec,\n    RetryPolicy,\n    RunArtifactStore,\n    ScriptedModelAdapter,\n    StopReason,\n    StandardTraceRedactor,\n    TextDelta,\n    Tool,\n    ToolCallDelta,\n    ToolResult,\n    UnsupportedRunSchemaVersionError,\n    Usage,\n    UsageUpdate,\n    create_session,\n    migrate_run_trace,\n)\n\n\ndef test_standard_trace_records_retries_tools_latency_usage_and_snapshot(\n    tmp_path: Path,\n) -> None:\n    class RetryingToolAdapter:\n        def __init__(self) -> None:\n            self.calls = 0\n\n        async def stream(self, request):\n            self.calls += 1\n            if self.calls == 1:\n                raise ModelAdapterError(\n                    ModelError(ModelErrorCode.SERVER, "temporary", True, status_code=503)\n                )\n            if self.calls == 2:\n                yield ToolCallDelta(0, "call-1", "echo", \'{"text":"hello"}\')\n                yield ModelEnd(StopReason.TOOL_USE)\n                return\n            yield TextDelta("finished")\n            yield UsageUpdate(Usage(7, 2, 9))\n            yield ModelEnd(StopReason.COMPLETE)\n\n    async def echo(arguments: dict[str, object]) -> ToolResult:\n        return ToolResult(str(arguments["text"]))\n\n    async def no_sleep(delay: float) -> None:\n        return None\n\n    workspace = tmp_path / "workspace"\n    storage = tmp_path / "data"\n    workspace.mkdir()\n    session = create_session(\n        RetryingToolAdapter(),\n        ModelSpec("scripted/observed"),\n        workspace=workspace,\n        storage_root=storage,\n        tools=(\n            Tool(\n                "echo",\n                "echo text",\n                {\n                    "type": "object",\n                    "properties": {"text": {"type": "string"}},\n                    "required": ["text"],\n                    "additionalProperties": False,\n                },\n                echo,\n            ),\n        ),\n        retry_policy=RetryPolicy(delays=(0,)),\n        sleeper=no_sleep,\n    )\n\n    asyncio.run(session.run("exercise trace"))\n\n    local = LocalWorkspace.open(workspace, storage_root=storage)\n    run = local.runs.list()[0]\n    records = local.runs.records(run.run_id)\n    header = records[0]\n    event_types = {\n        record.get("type") for record in records if record.get("record") == "event"\n    }\n    ending = records[-1]\n    assert header["snapshot"]["fingerprint"]\n    assert header["snapshot"]["platform"]["python"].startswith("3.11")\n    assert {"model_attempt_failed", "retry_scheduled", "tool_call_start", "tool_call_end"} <= event_types\n    assert any(record.get("tool_arguments") == {"text": "hello"} for record in records)\n    assert any(record.get("record") == "first_token" for record in records)\n    assert ending["record"] == "run_end"\n    assert ending["attempts"] == 3\n    assert ending["usage_provenance"] == "provider"\n\n\ndef test_run_store_keeps_append_only_annotations_and_content_addressed_artifacts(\n    tmp_path: Path,\n) -> None:\n    workspace = tmp_path / "workspace"\n    storage = tmp_path / "data"\n    workspace.mkdir()\n    session = create_session(\n        ScriptedModelAdapter(\n            [TextDelta("done"), ModelEnd(StopReason.COMPLETE)]\n        ),\n        ModelSpec("scripted/layout"),\n        workspace=workspace,\n        storage_root=storage,\n    )\n    asyncio.run(session.run("persist layout"))\n    local = LocalWorkspace.open(workspace, storage_root=storage)\n    run_id = local.runs.list()[0].run_id\n\n    first = local.runs.put_artifact(run_id, b"same content")\n    second = local.runs.put_artifact(run_id, b"same content")\n    local.runs.annotate(run_id, "review.score", {"score": 1})\n    local.runs.annotate(run_id, "review.note", {"note": "kept"})\n\n    run_directory = local.runs.directory_for(run_id)\n    assert first == second\n    assert len(list((run_directory / "artifacts" / "sha256").iterdir())) == 1\n    assert len(local.runs.annotations(run_id)) == 2\n    assert local.sessions.path_for(session.session_id).parent == local.root / "sessions"\n    assert local.runs.trace_path(run_id) == run_directory / "trace.jsonl"\n    assert local.runs.annotation_path(run_id) == run_directory / "annotations.jsonl"\n\n\ndef test_run_trace_recovery_and_versions_are_actionable_and_nondestructive(\n    tmp_path: Path,\n) -> None:\n    workspace = tmp_path / "workspace"\n    storage = tmp_path / "data"\n    workspace.mkdir()\n    session = create_session(\n        ScriptedModelAdapter(\n            [TextDelta("done"), ModelEnd(StopReason.COMPLETE)]\n        ),\n        ModelSpec("scripted/versioned"),\n        workspace=workspace,\n        storage_root=storage,\n    )\n    asyncio.run(session.run("version me"))\n    local = LocalWorkspace.open(workspace, storage_root=storage)\n    run_id = local.runs.list()[0].run_id\n    source = local.runs.trace_path(run_id)\n    original = source.read_bytes()\n    migrated = tmp_path / "migrated" / "trace.jsonl"\n\n    assert migrate_run_trace(source, migrated) == migrated.resolve()\n    assert source.read_bytes() == original\n    assert migrated.read_bytes() == original\n\n    source.write_bytes(original + b\'{"incomplete"\')\n    assert local.runs.summary(run_id).evidence_complete is False\n    source.write_bytes(original.replace(b\'"major":1\', b\'"major":99\', 1))\n    with pytest.raises(UnsupportedRunSchemaVersionError, match="migrate_run_trace"):\n        local.runs.records(run_id)\n\n\ndef test_artifact_redaction_preserves_full_sanitized_output(tmp_path: Path) -> None:\n    workspace = tmp_path / "workspace"\n    workspace.mkdir()\n    local = LocalWorkspace.open(\n        workspace,\n        storage_root=tmp_path / "data",\n        redactor=StandardTraceRedactor(secrets=("secret",), preview_chars=8),\n    )\n    local.runs.start("run-1", session_id=None, snapshot={})\n    artifacts = RunArtifactStore(local.runs)\n    artifacts.activate("run-1")\n    content = "prefix-secret-" + ("x" * 5000)\n\n    reference = artifacts.put_text(content)\n\n    digest = reference.removeprefix("sha256:")\n    retained = (\n        local.runs.directory_for("run-1") / "artifacts" / "sha256" / digest\n    ).read_text(encoding="utf-8")\n    assert "secret" not in retained\n    assert "[REDACTED]" in retained\n    assert retained.endswith("x" * 5000)\n'

In [ ]:
TEST_CONTRACT_SOURCE = 'from __future__ import annotations\n\nfrom pathlib import Path\nimport tomllib\n\nimport agent_harness\n\n\ndef test_chapter_09_freezes_public_factories_adapters_stores_and_policies() -> None:\n    expected = {\n        "AgentRuntime",\n        "AgentSession",\n        "CompactionPolicy",\n        "JSONLRunStore",\n        "JSONLSessionStore",\n        "ModelSpec",\n        "NonTerminalAdapter",\n        "OpenAICompatibleAdapter",\n        "OpenAICompatibleConfig",\n        "RetryPolicy",\n        "RunGuard",\n        "RunResult",\n        "StandardTraceSink",\n        "ToolExecutor",\n        "create_coding_session",\n        "create_session",\n    }\n\n    assert expected <= set(agent_harness.__all__)\n\n\ndef test_runtime_dependencies_stay_inside_the_accepted_small_boundary() -> None:\n    checkpoint = Path(__file__).parents[1]\n    project = tomllib.loads((checkpoint / "pyproject.toml").read_text(encoding="utf-8"))\n    dependencies = project["project"]["dependencies"]\n    names = {dependency.split("<", 1)[0].split(">", 1)[0].split("=", 1)[0] for dependency in dependencies}\n\n    assert names == {"jsonschema", "openai", "platformdirs"}\n    assert not any(\n        excluded in dependency.casefold()\n        for dependency in dependencies\n        for excluded in ("jupyter", "pytest", "textual", "mcp", "rpc")\n    )\n\n\ndef test_checkpoint_documentation_locks_privacy_output_and_v1_exclusions() -> None:\n    readme = (Path(__file__).parents[1] / "README.md").read_text(encoding="utf-8")\n\n    assert "There is no\\nAPI-key flag" in readme\n    assert "disables Session, Trace, Annotation, and Artifact persistence" in readme\n    assert "always ends with\\n`run_end`" in readme\n    for exclusion in (\n        "full TUI",\n        "RPC/server mode",\n        "MCP",\n        "subagents",\n        "multimodal content",\n        "optimizer",\n        "remote Extension installation",\n        "model catalog",\n        "built-in sandbox",\n    ):\n        assert exclusion in readme\n'

In [ ]:
durable_root = Path(temporary_workspace.name) / 'evidence'
observed = chapter9.create_session(
    chapter9.ScriptedModelAdapter([
        chapter9.TextDelta('durable evidence'),
        chapter9.ModelEnd(chapter9.StopReason.COMPLETE),
    ]),
    chapter9.ModelSpec('scripted/ch09-durable'),
    workspace=workspace, storage_root=durable_root,
)
durable_result = await observed.run('retain this run')
local = chapter9.LocalWorkspace.open(workspace, storage_root=durable_root)
assert observed.session_id is not None
assert local.sessions.read(observed.session_id).history()
assert local.runs.list(session_id=observed.session_id)[0].evidence_complete

## Observable Trace

The standard Trace is independent of resumable Session state. It records ordered Runtime Events, retry/tool/usage timing, frozen snapshot and platform evidence, then a semantic run_end record.

In [ ]:
run_summary = local.runs.list(session_id=observed.session_id)[0]
trace_records = local.runs.records(run_summary.run_id)
assert trace_records[0]['record'] == 'run_start'
assert trace_records[-1]['record'] == 'run_end'
assert trace_records[-1]['status'] == 'completed'
(run_summary.run_id, len(trace_records))

## Failure Boundaries and Trade-offs

`--no-save` disables every persistence family together. Standard Trace failures mark evidence incomplete without changing semantic success; strict tracing may promote that failure. Readers tolerate one incomplete final JSONL line and reject unsupported majors with migration guidance. No local evidence is uploaded implicitly.

In [ ]:
redactor = chapter9.StandardTraceRedactor(secrets=['chapter-secret'], preview_chars=32)
redacted = redactor.redact({'authorization': 'Bearer x', 'text': 'chapter-secret'})
assert redacted == {'authorization': '[REDACTED]', 'text': '[REDACTED]'}
assert not (Path(temporary_workspace.name) / 'no-save-data').exists()

## Checkpoint Export and Verification

The notebook exports the complete cumulative package, runs compile, offline install, import and cumulative tests, then generates the production `src/agent_harness` only from that verified Checkpoint.

In [ ]:
sys.path.remove(temporary_package.name)
temporary_package.cleanup()
temporary_workspace.cleanup()
for module_name in tuple(sys.modules):
    if module_name == 'agent_harness' or module_name.startswith('agent_harness.'):
        del sys.modules[module_name]

In [ ]:
from course.tools.checkpoint import (
    checkpoint_drift, export_checkpoint, export_production_package, production_drift,
)

chapter_path = ROOT / 'course' / 'notebooks' / '09_interfaces_and_release.ipynb'
checkpoint_path = ROOT / 'course' / 'checkpoints' / 'ch09'
checkpoint_result = export_checkpoint(chapter_path, checkpoint_path)
assert checkpoint_result.gates == ('compile', 'install', 'import', 'tests')
exported_production = export_production_package(checkpoint_path, ROOT)
assert checkpoint_drift(chapter_path, checkpoint_path) == ()
assert production_drift(checkpoint_path, ROOT) == ()
assert 'src/agent_harness/cli.py' in exported_production
checkpoint_result

## Public API Summary

Version one exposes `create_session`, `create_coding_session`, `AgentSession.run/start`, `RunResult`, `EventEnvelope`, `NonTerminalAdapter`, explicit Model/Tool/Store/Policy seams, and the `omega exec`, `omega chat`, `omega sessions`, and `omega runs` adapters.

In [ ]:
public_chapter9 = (
    'create_session', 'create_coding_session', 'AgentSession',
    'RunResult', 'EventEnvelope', 'NonTerminalAdapter',
    'JSONLSessionStore', 'JSONLRunStore', 'StandardTraceSink',
)
public_chapter9